# 11 · Autoencoders and Variational Autoencoders

**Part V — Generative models** · Estimated time: ~120 min

Up to now every model in this course was trained to predict a label, a next token or a target value. This notebook opens Part V by asking a different question: *what can a network learn from the inputs alone?* We will build autoencoders that compress images into a handful of numbers and reconstruct them, discover that the linear version recovers the subspace found by principal component analysis, use reconstruction error to find anomalies, and then confront the fact that a plain autoencoder is a surprisingly bad generator. That failure motivates the **variational autoencoder** (VAE), which we derive from first principles — the intractable marginal likelihood, the evidence lower bound, the reparameterization trick and the closed-form KL term — before implementing it, probing its latent space and measuring how good a generator it is.

## Learning objectives

- Explain representation learning and the manifold hypothesis, and quantify the intrinsic dimension of MNIST with a PCA spectrum.
- Implement linear and convolutional autoencoders in PyTorch and verify numerically that a linear autoencoder spans the PCA subspace.
- Use bottleneck size, denoising and latent-space visualizations to understand what an autoencoder learns, and apply reconstruction error to anomaly detection with a ROC analysis.
- Derive the evidence lower bound (ELBO), the reparameterization trick and the closed-form Gaussian KL divergence step by step.
- Implement and train a VAE and a conditional VAE on MNIST, monitor the reconstruction and KL terms separately, and generate samples, latent manifolds and interpolations.
- Diagnose posterior collapse, apply KL annealing, and interpret a beta sweep as a rate–distortion trade-off.
- Evaluate generative models with a feature-space Fréchet distance and a linear probe on the learned latents.
- Describe the VQ-VAE idea and how VAE latent spaces are used by latent diffusion models.

## Prerequisites

- Notebook 01 · Mathematical Foundations (PCA/SVD, expectation, KL divergence, maximum likelihood)
- Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization (Adam, regularization, reading loss curves)
- Notebook 05 · The PyTorch Workflow and Best Practices (modules, training loops, evaluation)
- Notebook 06 · Convolutional Neural Networks (convolutions, transposed convolutions)

## Contents

1. Representation learning and the manifold hypothesis
2. Autoencoders
3. Why plain autoencoders are poor generators
4. Variational autoencoders: the theory
5. A VAE on MNIST
6. Conditional VAE
7. Evaluating generative models
8. Applications and modern context

followed by Exercises, Summary and Further reading.

In [ ]:
import math, os, random, time, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets
from torchvision.utils import make_grid
from scipy import linalg
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import roc_auc_score, roc_curve

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

# FAST_RUN = True shrinks every training budget so the whole notebook runs in a few
# minutes on a CPU. Set it to False for the full experiments (recommended on a GPU).
FAST_RUN = False
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

# Budgets used throughout the notebook (every training loop reads from these).
N_TRAIN = 10_000 if FAST_RUN else 60_000    # MNIST training images kept in memory
EPOCHS_AE = 1 if FAST_RUN else 8            # plain autoencoders
EPOCHS_VAE = 2 if FAST_RUN else 15          # variational autoencoders
WIDTH = 16 if FAST_RUN else 32              # base channel width of the conv encoders / decoders
BATCH_SIZE = 128
print(f"FAST_RUN={FAST_RUN}: N_TRAIN={N_TRAIN}, EPOCHS_AE={EPOCHS_AE}, "
      f"EPOCHS_VAE={EPOCHS_VAE}, WIDTH={WIDTH}")

### Data: MNIST and FashionMNIST as in-memory tensors

Both datasets are tiny (60,000 + 10,000 grey-scale images of 28 × 28 pixels each), so instead of a `DataLoader` with per-item transforms we load each split **once** into a float tensor of shape `(N, 1, 28, 28)` with pixel values in $[0, 1]$ and move it to the device. Mini-batches are then just index slices of that tensor, which is far faster than a Python `Dataset` when the model itself is small. FashionMNIST is loaded only as a *test* set: it will play the role of "anomalous" inputs in Section 2.7 and never enters training.

The `torchvision` datasets download into `DATA_DIR` (`../data`) on first use and are read from disk afterwards.

In [ ]:
def load_split(dataset_cls, train):
    """Load a torchvision MNIST-style dataset once and return float tensors.

    Returns X of shape (N, 1, 28, 28) with values in [0, 1] and integer labels y of shape (N,).
    """
    ds = dataset_cls(DATA_DIR, train=train, download=True)
    X = ds.data.unsqueeze(1).float().div(255.0)
    y = ds.targets.clone()
    return X, y

t0 = time.time()
X_train_full, y_train_full = load_split(datasets.MNIST, train=True)
X_test, y_test = load_split(datasets.MNIST, train=False)
X_fashion, y_fashion = load_split(datasets.FashionMNIST, train=False)
print(f"loaded MNIST train {tuple(X_train_full.shape)}, test {tuple(X_test.shape)} and "
      f"FashionMNIST test {tuple(X_fashion.shape)} in {time.time() - t0:.1f}s")

# A fixed random subset of the training set, moved to the device once.
subset = torch.randperm(X_train_full.shape[0])[:N_TRAIN]
X_train = X_train_full[subset].to(device)
y_train = y_train_full[subset].to(device)
X_test, y_test = X_test.to(device), y_test.to(device)
X_fashion, y_fashion = X_fashion.to(device), y_fashion.to(device)
print(f"X_train {tuple(X_train.shape)} {X_train.dtype} on {X_train.device}, "
      f"pixel range [{X_train.min().item():.0f}, {X_train.max().item():.0f}]")

A small display helper is used throughout: it takes a batch `(N, 1, 28, 28)`, tiles it with `torchvision.utils.make_grid`, and shows the result with a grey colormap. A second helper, `show_rows`, stacks several such batches with a text label per row, which is convenient for "original / reconstruction" comparisons.

In [ ]:
def show_images(x, nrow=10, title=None, figsize=(7, 3)):
    """Display a batch of images (N, 1, 28, 28) with values in [0, 1] as a grid."""
    x = x.detach().float().cpu().clamp(0, 1)
    grid = make_grid(x, nrow=nrow, padding=1, pad_value=0.5)
    plt.figure(figsize=figsize)
    plt.imshow(grid[0].cpu().numpy(), cmap="gray", vmin=0, vmax=1)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.tight_layout(); plt.show()

def show_rows(rows, labels, figsize=(7, 5)):
    """Display several batches of images (one batch per row) with a text label on the left."""
    fig, axes = plt.subplots(len(rows), 1, figsize=figsize)
    for ax, imgs, lab in zip(np.atleast_1d(axes), rows, labels):
        grid = make_grid(imgs.detach().float().cpu().clamp(0, 1), nrow=imgs.shape[0],
                         padding=1, pad_value=0.5)
        ax.imshow(grid[0].cpu().numpy(), cmap="gray", vmin=0, vmax=1)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        ax.set_ylabel(lab, rotation=0, ha="right", va="center", fontsize=9)
    plt.tight_layout(); plt.show()

show_images(X_train[:20], nrow=10, title="Twenty training digits")
print("labels:", y_train[:20].tolist())

## 1. Representation learning and the manifold hypothesis

A 28 × 28 grey-scale image is a point in $\mathbb{R}^{784}$. If you sample such a point uniformly at random — every pixel an independent random intensity — you get static, never a digit. Digits are an astronomically small, highly structured subset of pixel space. The **manifold hypothesis** states this precisely: natural data concentrate near a low-dimensional, curved surface (a *manifold*) embedded in the high-dimensional input space. A "3" can be thickened, slanted, translated or drawn with a rounder top, and each of these variations is a smooth direction *along* the manifold; moving *off* the manifold quickly produces images that are not digits at all.

**Representation learning** is the attempt to find coordinates for this manifold: a mapping $x \mapsto z$ into a lower-dimensional space in which the meaningful factors of variation become explicit. Good coordinates make downstream tasks easier (a linear classifier on $z$ may do what a deep network needs on $x$), make the data compressible, and — if we can also map back from $z$ to $x$ — let us *generate* new data by picking new coordinates.

How do we learn such coordinates without labels? Three regimes are worth distinguishing:

- **Supervised** representation learning: the hidden layers of a classifier are representations, but they are shaped by the label and discard everything irrelevant to it (a digit classifier has no reason to encode stroke thickness).
- **Unsupervised** learning in the classical sense: fit a model of the data distribution $p(x)$ itself, or of a compression of it. Autoencoders and VAEs belong here.
- **Self-supervised** learning: invent a prediction task from the data alone (predict a masked patch, tell whether two augmented views come from the same image, predict the next token). Notebook 07 · Transfer Learning and Modern Vision sketched contrastive methods, and Notebook 09 · Attention and Transformers trained a language model, which is self-supervision on text. Autoencoders can be read as the earliest self-supervised task: predict the input from a compressed version of itself.

> **Key idea.** Compression forces abstraction. If a network must reconstruct a 784-pixel digit from 16 numbers, those 16 numbers have to encode *what kind of digit it is and how it is drawn*, because there is no room for anything else.

Before we build anything, let us check the hypothesis on MNIST with a tool from Notebook 01 · Mathematical Foundations: the spectrum of principal components. If digits really lived in all 784 dimensions equally, the explained variance would be spread evenly over the components, as it is for pixel noise. If they lie near a low-dimensional subspace, a few dozen components will capture most of the variance.

In [ ]:
n_pca = 5_000
X_flat = X_train[:n_pca].reshape(n_pca, -1)                     # (5000, 784)
X_flat_c = X_flat - X_flat.mean(dim=0, keepdim=True)
var_mnist = torch.linalg.svdvals(X_flat_c) ** 2 / (n_pca - 1)   # variance along each principal axis

torch.manual_seed(SEED)
noise = torch.rand_like(X_flat)                                 # "images" made of uniform pixel noise
noise_c = noise - noise.mean(dim=0, keepdim=True)
var_noise = torch.linalg.svdvals(noise_c) ** 2 / (n_pca - 1)

cum_mnist = (torch.cumsum(var_mnist, 0) / var_mnist.sum()).cpu().numpy()
cum_noise = (torch.cumsum(var_noise, 0) / var_noise.sum()).cpu().numpy()
k95_mnist = int(np.searchsorted(cum_mnist, 0.95)) + 1
k95_noise = int(np.searchsorted(cum_noise, 0.95)) + 1

plt.plot(cum_mnist, label=f"MNIST (95% of variance in {k95_mnist} components)")
plt.plot(cum_noise, label=f"uniform pixel noise (95% in {k95_noise} components)")
plt.axhline(0.95, color="gray", ls="--", lw=1)
plt.xlabel("number of principal components"); plt.ylabel("cumulative explained variance")
plt.title("MNIST is far from filling its 784 dimensions"); plt.legend()
plt.tight_layout(); plt.show()

The curves could hardly be more different. For pixel noise the variance is spread almost uniformly, so you need nearly all 784 components to explain 95 % of it. For MNIST a few dozen components already explain most of the variance and around 150 explain 95 %. Even this overstates the intrinsic dimension: PCA can only find *linear* subspaces, and the digit manifold is curved. A non-linear model that follows the curvature can do with far fewer coordinates — that is exactly what the convolutional autoencoders below will demonstrate when a 2-dimensional or 8-dimensional bottleneck reconstructs digits that PCA with the same number of components cannot.

## 2. Autoencoders

An **autoencoder** (AE) is a pair of networks trained together: an **encoder** $f_\phi : \mathbb{R}^D \to \mathbb{R}^d$ that maps an input to a **code** (or **latent vector**) $z = f_\phi(x)$, and a **decoder** $g_\theta : \mathbb{R}^d \to \mathbb{R}^D$ that maps the code back to a reconstruction $\hat{x} = g_\theta(z)$. Training minimizes a reconstruction loss such as

$$
\mathcal{L}(\theta, \phi) = \frac{1}{N} \sum_{i=1}^{N} \ell\big(x_i,\; g_\theta(f_\phi(x_i))\big),
$$

with $\ell$ the squared error or a per-pixel cross-entropy. If $d \ge D$ and the networks are flexible, the trivial solution "copy the input" achieves zero loss and nothing is learned. The interesting case is the **undercomplete** autoencoder with $d \ll D$: the **bottleneck** forces the code to keep only what matters most for reconstruction. Everything we do in this section is a study of what that pressure produces.

### 2.1 The linear autoencoder recovers PCA

Consider the simplest possible autoencoder: a linear encoder $z = W_e x$ with $W_e \in \mathbb{R}^{k \times D}$, a linear decoder $\hat{x} = W_d z$ with $W_d \in \mathbb{R}^{D \times k}$, no biases, no activation functions, trained with the squared error on centred data. The reconstruction $\hat{x} = W_d W_e x$ is a linear map of rank at most $k$, so the loss is

$$
\mathcal{L}(W_d, W_e) = \frac{1}{N} \sum_i \| x_i - W_d W_e x_i \|^2 .
$$

A classical result (Eckart–Young for the matrix statement; Baldi and Hornik, 1989, for the autoencoder formulation) says that the minimum over all rank-$k$ linear maps is achieved by the **orthogonal projection onto the top-$k$ principal subspace**: $W_d W_e = V_k V_k^\top$, where the columns of $V_k$ are the $k$ leading eigenvectors of the data covariance. Moreover, *every other critical point of this loss is a saddle*, so gradient descent is expected to find the global optimum. Two remarks matter for interpreting the experiment:

1. The optimum fixes only the *product* $W_d W_e$. Any invertible $k \times k$ matrix $A$ gives another optimum $(W_d A^{-1}, A W_e)$. The columns of $W_d$ therefore span the principal subspace but are in general **not** the principal components themselves: they are a rotated, scaled, non-orthogonal basis of the same subspace.
2. Nothing orders the coordinates by variance. PCA gives you component 1, 2, 3, …; the autoencoder gives you an arbitrary basis of the same $k$-dimensional space.

So the right test is not "are the weights equal to the principal components" but "do the *subspaces* coincide". Two subspaces are compared through their **principal angles**: if $Q$ and $V_k$ are orthonormal bases, the singular values of $Q^\top V_k$ are the cosines of the principal angles, and they all equal 1 exactly when the subspaces are the same. We will also compare the projection matrices $Q Q^\top$ and $V_k V_k^\top$ directly, and compare the product $W_d W_e$ against $V_k V_k^\top$.

We train on 2,000 centred images by full-batch gradient descent (the whole dataset fits in one matrix product) with Adam and a cosine learning-rate decay so that the iterate settles precisely instead of jittering around the optimum.

In [ ]:
class LinearAE(nn.Module):
    """A linear autoencoder: x -> W_e x -> W_d W_e x, with no biases and no non-linearities."""

    def __init__(self, d_in, k):
        super().__init__()
        self.enc = nn.Linear(d_in, k, bias=False)
        self.dec = nn.Linear(k, d_in, bias=False)

    def forward(self, x):
        return self.dec(self.enc(x))

K_LIN, N_LIN = 6, 2_000
X_lin = X_train[:N_LIN].reshape(N_LIN, -1)
X_lin = X_lin - X_lin.mean(dim=0, keepdim=True)      # PCA assumes centred data

torch.manual_seed(SEED)
lin_ae = LinearAE(784, K_LIN).to(device)
STEPS_LIN = 1_000 if FAST_RUN else 3_000
opt = torch.optim.Adam(lin_ae.parameters(), lr=1e-2)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=STEPS_LIN)
t0 = time.time()
for step in range(1, STEPS_LIN + 1):
    loss = F.mse_loss(lin_ae(X_lin), X_lin)           # full-batch gradient descent
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
    if step % (STEPS_LIN // 5) == 0:
        print(f"step {step:5d} | reconstruction MSE {loss.item():.5f} | {time.time() - t0:.1f}s")

Now the numerical verification. PCA is computed directly from the singular value decomposition of the centred data matrix (the right singular vectors are the principal directions). For the autoencoder we orthonormalize the decoder columns with a QR decomposition to obtain a basis $Q$ of the learned subspace.

In [ ]:
with torch.no_grad():
    _, S_lin, Vh = torch.linalg.svd(X_lin, full_matrices=False)
    V_k = Vh[:K_LIN].T                                       # (784, k): top-k principal directions
    Q, _ = torch.linalg.qr(lin_ae.dec.weight)                # dec.weight is (784, k); Q spans the AE subspace
    cosines = torch.linalg.svdvals(Q.T @ V_k)                # cosines of the principal angles
    P_ae, P_pca = Q @ Q.T, V_k @ V_k.T                       # projection matrices (784, 784)
    proj_gap = torch.linalg.norm(P_ae - P_pca) / torch.linalg.norm(P_pca)
    W_map = lin_ae.dec.weight @ lin_ae.enc.weight            # the full linear map W_d W_e
    map_gap = torch.linalg.norm(W_map - P_pca) / torch.linalg.norm(P_pca)
    mse_ae = F.mse_loss(lin_ae(X_lin), X_lin).item()
    mse_pca = F.mse_loss(X_lin @ P_pca, X_lin).item()

print("cosines of the principal angles between the AE and PCA subspaces:",
      np.round(cosines.cpu().numpy(), 4))
print(f"relative Frobenius gap between projection matrices Q Q^T and V_k V_k^T : {proj_gap.item():.4f}")
print(f"relative Frobenius gap between W_d W_e and V_k V_k^T                 : {map_gap.item():.4f}")
print(f"reconstruction MSE: linear AE {mse_ae:.5f} | PCA-{K_LIN} {mse_pca:.5f} | ratio {mse_ae / mse_pca:.4f}")

fig, axes = plt.subplots(2, K_LIN, figsize=(7, 2.8))
for j in range(K_LIN):
    axes[0, j].imshow(V_k[:, j].reshape(28, 28).cpu().numpy(), cmap="RdBu_r")
    axes[1, j].imshow(Q[:, j].reshape(28, 28).cpu().numpy(), cmap="RdBu_r")
    for ax in (axes[0, j], axes[1, j]):
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
axes[0, 0].set_ylabel("PCA", fontsize=9); axes[1, 0].set_ylabel("AE (QR)", fontsize=9)
plt.suptitle("Principal components (top) and an orthonormal basis of the AE decoder subspace (bottom)", fontsize=9)
plt.tight_layout(); plt.show()

All cosines are close to 1, the projection matrices agree to within a percent or so, and the reconstruction error of the trained linear autoencoder matches PCA with the same number of components. If one cosine lags slightly behind, it is the direction with the smallest gap between consecutive eigenvalues — the least "identifiable" direction, and the last to converge. The pictures make the earlier remark concrete: the basis found by the autoencoder looks nothing like the ordered principal components, yet it spans the same six-dimensional subspace, which is all the loss can see.

> **Key idea.** A linear autoencoder with squared error learns the PCA subspace and nothing more; every gain of non-linear autoencoders comes from *curving* the subspace to follow the data manifold.

### 2.2 A convolutional autoencoder on MNIST

To follow the curvature we need non-linear encoders and decoders. For images the natural choice is convolutional. Our encoder halves the spatial resolution twice with stride-2 convolutions ($28 \to 14 \to 7$), flattens, and applies a linear layer to the code size $d$. The decoder mirrors it: a linear layer expands the code to a $7 \times 7$ feature map, and two **transposed convolutions** (Notebook 06 · Convolutional Neural Networks) upsample back to $28 \times 28$. With kernel size 4, stride 2 and padding 1 each transposed convolution exactly doubles the resolution.

The decoder outputs **logits**, one per pixel. We train with the per-pixel **binary cross-entropy** between the sigmoid of these logits and the grey-level target, summed over the 784 pixels and averaged over the batch. Two reasons for this choice rather than the squared error: it treats each pixel as a probability of "ink", which matches the nearly binary nature of MNIST and gives sharper reconstructions, and it is precisely the negative log-likelihood of the Bernoulli decoder that we will derive in Section 4.5, so the numbers reported here are directly comparable with the VAE's reconstruction term later. We report the squared error too, because it is the more familiar unit and because Section 2.7 uses it as an anomaly score.

In [ ]:
class ConvEncoder(nn.Module):
    """28x28 image -> vector of size out_dim (the VAE will ask for 2*latent_dim outputs)."""

    def __init__(self, out_dim, width=WIDTH, in_channels=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, width, 4, stride=2, padding=1), nn.ReLU(),   # 28 -> 14
            nn.Conv2d(width, 2 * width, 4, stride=2, padding=1), nn.ReLU(),     # 14 -> 7
            nn.Flatten(),
        )
        self.fc = nn.Linear(2 * width * 7 * 7, out_dim)

    def forward(self, x):
        return self.fc(self.conv(x))


class ConvDecoder(nn.Module):
    """vector of size in_dim -> 28x28 map of pixel logits."""

    def __init__(self, in_dim, width=WIDTH):
        super().__init__()
        self.width = width
        self.fc = nn.Linear(in_dim, 2 * width * 7 * 7)
        self.deconv = nn.Sequential(
            nn.ReLU(),
            nn.ConvTranspose2d(2 * width, width, 4, stride=2, padding=1), nn.ReLU(),   # 7 -> 14
            nn.ConvTranspose2d(width, 1, 4, stride=2, padding=1),                      # 14 -> 28
        )

    def forward(self, z):
        h = self.fc(z).view(z.shape[0], 2 * self.width, 7, 7)
        return self.deconv(h)


class ConvAE(nn.Module):
    """Deterministic convolutional autoencoder with a latent_dim-dimensional bottleneck."""

    def __init__(self, latent_dim, width=WIDTH):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = ConvEncoder(latent_dim, width)
        self.decoder = ConvDecoder(latent_dim, width)

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        return self.decode(self.encode(x))

probe_ae = ConvAE(8).to(device)
with torch.no_grad():
    out = probe_ae(X_train[:4])
    print("input", tuple(X_train[:4].shape), "-> code", tuple(probe_ae.encode(X_train[:4]).shape),
          "-> logits", tuple(out.shape))
print(f"parameters: {sum(p.numel() for p in probe_ae.parameters()):,}")

Next come the small, reusable training utilities: a mini-batch iterator over in-memory tensors, the two reconstruction metrics (BCE in nats per image, squared error per image), an evaluation function and a training loop that prints one line per epoch. The optional `corrupt` argument applies a noise function to the *input* while the loss is still computed against the *clean* image — that single argument turns the loop into a denoising autoencoder trainer in Section 2.5.

In [ ]:
def iterate_batches(tensors, batch_size, shuffle=True):
    """Yield tuples of aligned mini-batches sliced from tensors that already live on the device."""
    n = tensors[0].shape[0]
    dev = tensors[0].device
    idx = torch.randperm(n, device=dev) if shuffle else torch.arange(n, device=dev)
    for i in range(0, n, batch_size):
        j = idx[i:i + batch_size]
        yield tuple(t[j] for t in tensors)

def recon_bce(logits, x):
    """Binary cross-entropy between pixel logits and targets, summed over pixels: one value per image."""
    return F.binary_cross_entropy_with_logits(logits, x, reduction="none").flatten(1).sum(dim=1)

def recon_mse(logits, x):
    """Squared error between the decoded image sigmoid(logits) and x, summed over pixels, per image."""
    return (torch.sigmoid(logits) - x).pow(2).flatten(1).sum(dim=1)

@torch.no_grad()
def evaluate_ae(model, X, batch_size=1000, corrupt=None):
    """Mean per-image BCE (nats) and squared error of an autoencoder on a tensor of images."""
    model.eval()
    bce = mse = 0.0
    for (xb,) in iterate_batches([X], batch_size, shuffle=False):
        inp = xb if corrupt is None else corrupt(xb)
        logits = model(inp)
        bce += recon_bce(logits, xb).sum().item()
        mse += recon_mse(logits, xb).sum().item()
    return bce / X.shape[0], mse / X.shape[0]

def train_ae(model, X, epochs, lr=1e-3, batch_size=BATCH_SIZE, corrupt=None, verbose=True):
    """Train an autoencoder with Adam on the per-image BCE. Returns a per-epoch history dict."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_bce": [], "test_bce": [], "test_mse": []}
    for epoch in range(1, epochs + 1):
        model.train(); t0 = time.time(); running = 0.0
        for (xb,) in iterate_batches([X], batch_size):
            inp = xb if corrupt is None else corrupt(xb)
            loss = recon_bce(model(inp), xb).mean()
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            running += loss.item() * xb.shape[0]
        test_bce, test_mse = evaluate_ae(model, X_test, corrupt=corrupt)
        history["train_bce"].append(running / X.shape[0])
        history["test_bce"].append(test_bce); history["test_mse"].append(test_mse)
        if verbose:
            print(f"epoch {epoch:2d} | train BCE {running / X.shape[0]:7.2f} | test BCE {test_bce:7.2f} "
                  f"| test MSE {test_mse:6.2f} | {time.time() - t0:.1f}s")
    return history

### 2.3 Reconstruction quality versus bottleneck size

How much does the code size matter? We train three autoencoders with $d \in \{2, 8, 32\}$ and compare their test reconstruction error with PCA using the same number of components. PCA is computed from the eigendecomposition of the $784 \times 784$ pixel covariance of the training subset (cheaper than an SVD of the full data matrix and numerically equivalent for our purpose).

In [ ]:
LATENT_DIMS = [2, 8, 32]
aes, ae_hist = {}, {}
for d in LATENT_DIMS:
    torch.manual_seed(SEED)
    print(f"--- convolutional AE, latent dimension {d} ---")
    aes[d] = ConvAE(d).to(device)
    ae_hist[d] = train_ae(aes[d], X_train, EPOCHS_AE)

# PCA baseline from the pixel covariance of the training subset
X_flat_train = X_train.reshape(X_train.shape[0], -1)
pix_mean = X_flat_train.mean(dim=0, keepdim=True)
cov = (X_flat_train - pix_mean).T @ (X_flat_train - pix_mean) / (X_flat_train.shape[0] - 1)
evals, evecs = torch.linalg.eigh(cov)                       # ascending eigenvalues
evecs = evecs.flip(dims=[1])                                # column j = j-th principal direction

@torch.no_grad()
def pca_reconstruct(X, d):
    """Project images on the top-d principal directions and map back to pixel space."""
    V = evecs[:, :d]
    Xc = X.reshape(X.shape[0], -1) - pix_mean
    return (Xc @ V @ V.T + pix_mean).reshape_as(X)

pca_mse = {d: (pca_reconstruct(X_test, d) - X_test).pow(2).flatten(1).sum(1).mean().item()
           for d in LATENT_DIMS}
ae_mse = {d: ae_hist[d]["test_mse"][-1] for d in LATENT_DIMS}
for d in LATENT_DIMS:
    print(f"d={d:3d}: test squared error per image  conv AE {ae_mse[d]:6.2f}   PCA {pca_mse[d]:6.2f}")

plt.plot(LATENT_DIMS, [ae_mse[d] for d in LATENT_DIMS], "o-", label="convolutional AE")
plt.plot(LATENT_DIMS, [pca_mse[d] for d in LATENT_DIMS], "s--", label="PCA")
plt.xscale("log", base=2); plt.xlabel("bottleneck size d"); plt.ylabel("test squared error per image")
plt.title("Reconstruction error versus bottleneck size"); plt.legend()
plt.tight_layout(); plt.show()

The error falls steeply with the bottleneck size, and at every size the non-linear autoencoder beats PCA — by a wide margin for small $d$, where a curved 2-dimensional surface captures far more of the digit manifold than any plane through pixel space. Looking at actual reconstructions tells the story better than the numbers: PCA with two components produces a blurry "average digit", while the 2-dimensional convolutional code already yields recognizable, if generic, digits. With 32 dimensions both methods produce clearly recognizable reconstructions, but the autoencoder keeps fine stroke details that PCA still blurs.

In [ ]:
x8 = X_test[:8]
rows, labels = [x8], ["original"]
for d in LATENT_DIMS:
    rows.append(pca_reconstruct(x8, d)); labels.append(f"PCA d={d}")
    with torch.no_grad():
        aes[d].eval(); rows.append(torch.sigmoid(aes[d](x8)))
    labels.append(f"conv AE d={d}")
show_rows(rows, labels, figsize=(7, 6.5))

### 2.4 Looking into the latent space: a 2-D map of MNIST

The autoencoder with $d = 2$ maps every image to a point in the plane, so we can draw the whole test set and colour each point by its (unused during training) digit label. This is the first time we can *see* a learned representation.

In [ ]:
@torch.no_grad()
def encode_all(model, X, batch_size=1000):
    """Encode a tensor of images with a deterministic autoencoder, batch by batch."""
    model.eval()
    return torch.cat([model.encode(xb) for (xb,) in iterate_batches([X], batch_size, shuffle=False)])

z_ae2 = encode_all(aes[2], X_test).cpu().numpy()
y_test_np = y_test.cpu().numpy()

plt.figure(figsize=(6.5, 5))
sc = plt.scatter(z_ae2[:, 0], z_ae2[:, 1], c=y_test_np, cmap="tab10", s=3, alpha=0.6)
plt.colorbar(sc, ticks=range(10), label="digit")
plt.xlabel("$z_1$"); plt.ylabel("$z_2$")
plt.title("Test-set codes of the convolutional AE with d = 2")
plt.tight_layout(); plt.show()

Although the network never saw a label, the digit classes form distinct clusters, and related digits (4 and 9, 3 and 5 and 8) sit next to each other because they share strokes. Two features of the picture will matter later: the codes spread over an irregular region with a scale dictated by nothing in particular (the encoder is free to use any range), and there are visible **gaps** between and around the clusters. Section 3 shows what the decoder does with points in those gaps.

A second way to look at a latent space is to walk through it. Take two test images $x_a$ and $x_b$, encode them, and decode the points on the straight line $z(t) = (1 - t) z_a + t z_b$. Compare this with the naive alternative of cross-fading the pixels: the pixel-space path passes through images that are superpositions of two digits, whereas the latent-space path should morph one digit into the other through plausible intermediate shapes.

In [ ]:
@torch.no_grad()
def interpolate_ae(model, xa, xb, steps=10):
    """Decode the straight line between the codes of two images."""
    model.eval()
    za, zb = model.encode(xa[None]), model.encode(xb[None])
    t = torch.linspace(0, 1, steps, device=za.device)[:, None]
    return torch.sigmoid(model.decode((1 - t) * za + t * zb))

idx_a = (y_test == 3).nonzero()[0].item()
idx_b = (y_test == 8).nonzero()[0].item()
xa, xb = X_test[idx_a], X_test[idx_b]
t = torch.linspace(0, 1, 10, device=device)[:, None, None, None]
pixel_path = (1 - t) * xa[None] + t * xb[None]
show_rows([pixel_path, interpolate_ae(aes[32], xa, xb), interpolate_ae(aes[2], xa, xb)],
          ["pixel space", "latent, d=32", "latent, d=2"], figsize=(7, 3))

The pixel cross-fade is a double exposure: the 3 fades out while the 8 fades in. The 32-dimensional latent walk is better but still shows ghosting halfway, because the straight line leaves the region where the decoder has seen codes. The 2-dimensional walk morphs through digit-like shapes at every step — a lower-dimensional latent space is denser, so straight lines are more likely to stay on the manifold. This trade-off between reconstruction fidelity (higher $d$) and a well-behaved latent space (lower $d$) is the tension the VAE will resolve.

### 2.5 Denoising autoencoders

A **denoising autoencoder** (Vincent et al., 2008) is trained to reconstruct the clean input $x$ from a corrupted version $\tilde{x} = C(x)$, e.g. with additive Gaussian noise or randomly zeroed pixels:

$$
\mathcal{L} = \mathbb{E}_{x}\, \mathbb{E}_{\tilde{x} \sim C(\cdot \mid x)} \big[ \ell(x, g_\theta(f_\phi(\tilde{x}))) \big].
$$

The noise plays two roles. Practically, it is a strong regularizer: the identity function is no longer a solution, so even an overcomplete network must learn structure. Conceptually, denoising is *manifold learning*: corrupted inputs lie off the data manifold, and the network learns a vector field that points from off-manifold points back onto it. This second reading is remarkably deep — Vincent (2011) showed that the optimal denoiser for small Gaussian noise computes the gradient of the log-density of the data, $\nabla_x \log p(x)$ (the *score*), and this identity is the seed from which diffusion models grow in Notebook 12 · GANs and Diffusion Models.

We corrupt with Gaussian noise of standard deviation 0.5 (clamped back into $[0, 1]$), train a denoising autoencoder with $d = 32$, and compare it with the plain autoencoder of the same size, which has never seen a noisy input.

In [ ]:
NOISE_STD = 0.5

def add_noise(x, std=NOISE_STD):
    """Corrupt images with additive Gaussian noise, clamped back into the valid pixel range."""
    return (x + std * torch.randn_like(x)).clamp(0, 1)

torch.manual_seed(SEED)
dae = ConvAE(32).to(device)
print("--- denoising conv AE, latent dimension 32 (inputs corrupted, targets clean) ---")
dae_hist = train_ae(dae, X_train, EPOCHS_AE, corrupt=add_noise)

torch.manual_seed(SEED + 1)
x_noisy = add_noise(X_test[:8])
with torch.no_grad():
    aes[32].eval(); dae.eval()
    rec_plain = torch.sigmoid(aes[32](x_noisy))
    rec_dae = torch.sigmoid(dae(x_noisy))
show_rows([X_test[:8], x_noisy, rec_plain, rec_dae],
          ["clean", "noisy input", "plain AE", "denoising AE"], figsize=(7, 4))

torch.manual_seed(SEED + 2)
bce_plain, mse_plain = evaluate_ae(aes[32], X_test, corrupt=add_noise)
torch.manual_seed(SEED + 2)
bce_dae, mse_dae = evaluate_ae(dae, X_test, corrupt=add_noise)
print(f"noisy test inputs -> clean targets: squared error per image  plain AE {mse_plain:.2f} | "
      f"denoising AE {mse_dae:.2f}")

The plain autoencoder faithfully reconstructs the noise it is given — reconstruction was its only objective, and with 32 dimensions it has room to encode some of the speckle. The denoising autoencoder returns clean digits, and its squared error against the clean target is much lower. Notice that it did not merely learn "blur the input": the strokes it produces are crisp, which it can only do by recognizing *which* digit is hidden under the noise and redrawing it. Denoising is a pretext task that forces semantic understanding.

> **Try it.** Raise `NOISE_STD` to 1.0 or replace the corruption with masking noise (`x * (torch.rand_like(x) > 0.5)`) and retrain. At what corruption level does the denoiser start to hallucinate the wrong digit?

### 2.6 Sparse and contractive autoencoders (in prose)

Two other classical ways of regularizing an autoencoder deserve a mention, because both ideas resurface elsewhere in the course.

A **sparse autoencoder** allows a wide, even overcomplete, code but penalizes its activity, for instance with an L1 penalty $\lambda \sum_j |z_j|$ or by constraining the average activation of each code unit to a small target $\rho$ through a KL penalty $\sum_j \mathrm{KL}(\rho \,\|\, \hat{\rho}_j)$. Only a few units may fire for any given input, which pushes each unit to specialize on a recurring pattern (stroke detectors on MNIST, edge detectors on natural images). Sparse dictionaries of this kind are today the main tool for interpreting the internal activations of large language models.

A **contractive autoencoder** (Rifai et al., 2011) penalizes the sensitivity of the code to the input through the squared Frobenius norm of the encoder's Jacobian, $\lambda \| \partial f_\phi(x) / \partial x \|_F^2$. The encoder is thereby encouraged to be locally flat — to *contract* its neighbourhood — in every direction except the ones needed for reconstruction, which are exactly the tangent directions of the data manifold. Denoising and contractive penalties are closely related: for small Gaussian noise, the denoising objective is equivalent to a contractive penalty on the reconstruction function (Alain and Bengio, 2014).

### 2.7 Autoencoders for anomaly detection

An autoencoder trained on one kind of data learns to reconstruct *that* kind of data well. Feed it something else and the reconstruction is poor, because the bottleneck has no coordinates for structures it never saw. This makes the reconstruction error a natural **anomaly score**: fit the autoencoder on "normal" data, then flag test inputs whose reconstruction error exceeds a threshold. No anomalous examples are needed at training time, which is what makes the method attractive in fraud detection, manufacturing inspection or medical imaging, where anomalies are rare and heterogeneous.

We use the $d = 32$ autoencoder trained on MNIST and score two test sets: the MNIST test images (normal) and the FashionMNIST test images (anomalies: trousers, sneakers and shirts at the same resolution). The **ROC curve** plots the true positive rate against the false positive rate as the threshold sweeps over all values, and its area (**ROC-AUC**) summarizes how well the score separates the two groups: 0.5 is chance, 1.0 is perfect separation. We compute it with `sklearn.metrics.roc_auc_score`.

In [ ]:
@torch.no_grad()
def reconstruction_errors(model, X, batch_size=1000):
    """Per-image squared reconstruction error of a deterministic autoencoder."""
    model.eval()
    return torch.cat([recon_mse(model(xb), xb)
                      for (xb,) in iterate_batches([X], batch_size, shuffle=False)])

err_mnist = reconstruction_errors(aes[32], X_test).cpu().numpy()
err_fashion = reconstruction_errors(aes[32], X_fashion).cpu().numpy()
scores = np.concatenate([err_mnist, err_fashion])
is_anomaly = np.concatenate([np.zeros_like(err_mnist), np.ones_like(err_fashion)])
auc_ae = roc_auc_score(is_anomaly, scores)
fpr, tpr, _ = roc_curve(is_anomaly, scores)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].hist(err_mnist, bins=60, alpha=0.7, density=True, label="MNIST test (normal)")
axes[0].hist(err_fashion, bins=60, alpha=0.7, density=True, label="FashionMNIST (anomalies)")
axes[0].set_xlabel("reconstruction error (squared, per image)"); axes[0].set_ylabel("density")
axes[0].set_title("Score distributions"); axes[0].legend()
axes[1].plot(fpr, tpr, label=f"reconstruction error (AUC = {auc_ae:.3f})")
axes[1].plot([0, 1], [0, 1], "--", color="gray", label="chance")
axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate")
axes[1].set_title("ROC curve"); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"ROC-AUC of the reconstruction error as an anomaly score: {auc_ae:.4f}")

top_mnist = torch.as_tensor(np.argsort(err_mnist)[-8:], device=device)
low_fashion = torch.as_tensor(np.argsort(err_fashion)[:8], device=device)
show_rows([X_test[top_mnist], X_fashion[low_fashion]],
          ["hardest MNIST", "easiest Fashion"], figsize=(7, 2.5))

The two score distributions barely overlap and the AUC is far above 0.9: the autoencoder is a competent anomaly detector out of the box. The failure cases are instructive. The MNIST digits with the *highest* error are thick, unusual or badly written — genuinely atypical inputs, which is arguably what an anomaly detector should flag. The FashionMNIST items with the *lowest* error are thin, sparse items such as trousers or sandals that contain little ink: an autoencoder that reconstructs a mostly black image with a few strokes has an easy job regardless of what the strokes depict. Reconstruction error measures *how much the input looks like something the model can draw*, which correlates with, but is not the same as, "how anomalous is this". Exercise 2 revisits the task with the VAE's likelihood bound as the score.

> **Common pitfall.** Reconstruction-error scores depend on the amount of signal in the input. Simple, low-energy inputs get low errors even when they are anomalous. Normalize the error by the input energy, or use a likelihood-based score, when your anomalies may be *simpler* than the normal data.

## 3. Why plain autoencoders are poor generators

An autoencoder gives us a decoder, and a decoder maps latent vectors to images. It is tempting to conclude that we have a generative model: pick a random $z$, decode it, obtain a new digit. Let us try that with the 2-dimensional model, where we can see exactly what happens. We sample points uniformly inside the bounding box of the test-set codes and decode them.

In [ ]:
z_codes = torch.as_tensor(z_ae2, device=device)
lo, hi = z_codes.min(dim=0).values, z_codes.max(dim=0).values
torch.manual_seed(SEED)
z_rand = lo + (hi - lo) * torch.rand(30, 2, device=device)
with torch.no_grad():
    aes[2].eval()
    x_rand = torch.sigmoid(aes[2].decode(z_rand))

fig, axes = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1.1, 1]})
axes[0].scatter(z_ae2[:, 0], z_ae2[:, 1], c=y_test_np, cmap="tab10", s=2, alpha=0.4)
axes[0].scatter(z_rand[:, 0].cpu().numpy(), z_rand[:, 1].cpu().numpy(), marker="x", c="black", s=40,
                label="random latent points")
axes[0].set_xlabel("$z_1$"); axes[0].set_ylabel("$z_2$"); axes[0].legend(loc="upper right")
axes[0].set_title("Where the random points fall")
grid = make_grid(x_rand.cpu(), nrow=10, padding=1, pad_value=0.5)
axes[1].imshow(grid[0].cpu().numpy(), cmap="gray", vmin=0, vmax=1); axes[1].axis("off")
axes[1].set_title("Decoded random points")
plt.tight_layout(); plt.show()

H, _, _ = np.histogram2d(z_ae2[:, 0], z_ae2[:, 1], bins=30)
print(f"fraction of empty cells in a 30x30 grid over the code bounding box: {(H == 0).mean():.2f}")

Some decoded points are fine digits — those that happen to land inside a cluster. Others are smeared hybrids or unreadable blobs: they come from the **holes**, regions of the latent plane that no training image maps to. A sizeable fraction of the bounding box is empty, and the decoder's behaviour there is undefined in the strict sense: nothing in the training objective ever constrained it. The problem gets *worse* in higher dimensions, where the volume of the box grows exponentially while the codes stay on a thin manifold. Let us sample from a Gaussian fitted to the 32-dimensional codes — the most generous choice of "random" available to us — and decode.

In [ ]:
z32 = encode_all(aes[32], X_test)
mu32, sd32 = z32.mean(dim=0), z32.std(dim=0)
torch.manual_seed(SEED)
z_rand32 = mu32 + sd32 * torch.randn(30, 32, device=device)
with torch.no_grad():
    x_rand32 = torch.sigmoid(aes[32].decode(z_rand32))
show_images(x_rand32, nrow=10, title="d = 32 autoencoder: decoded samples from a Gaussian fit to the codes")

Hardly any of these are digits. Two things are missing from the plain autoencoder, and naming them tells us what to fix:

1. **No prior.** Nothing tells us *where* in latent space the codes live. We had to estimate a bounding box or a Gaussian after the fact, and the estimate does not describe the thin, curved region the codes actually occupy.
2. **No smoothness.** The encoder maps each image to a single point and the decoder is only trained at those points. Between them it may do anything, and it does.

The variational autoencoder fixes both with one change of viewpoint. It declares a prior $p(z) = \mathcal{N}(0, I)$ up front, encodes each image not to a point but to a small Gaussian *cloud*, and adds a penalty that keeps the union of the clouds close to the prior. Decoding a sample from the prior then lands in a region the decoder was trained on, and the clouds overlap, so the decoder is forced to be smooth. Making this precise requires a little probability, which is the subject of the next section.

## 4. Variational autoencoders: the theory

### 4.1 A latent-variable model of images

The VAE (Kingma and Welling, 2014; Rezende, Mohamed and Wierstra, 2014) starts from a **generative story** rather than from a network: to produce an image, first draw a latent vector from a simple prior, then draw the image from a distribution whose parameters are computed from the latent vector by a neural network,

$$
z \sim p(z) = \mathcal{N}(0, I_d), \qquad x \sim p_\theta(x \mid z).
$$

The network inside $p_\theta(x \mid z)$ is the decoder; for MNIST it outputs one number per pixel that parameterizes, say, a Bernoulli distribution (Section 4.5). The model assigns to every image the **marginal likelihood**

$$
p_\theta(x) = \int p_\theta(x \mid z)\, p(z)\, dz,
$$

and *maximum likelihood* would fit $\theta$ by maximizing $\sum_i \log p_\theta(x_i)$ over the training set, exactly as we fit classifiers by maximizing the log-probability of the correct label (Notebook 01 · Mathematical Foundations). The trouble is the integral. It has no closed form because a neural network sits inside the integrand, and estimating it by Monte Carlo — sample $z$ from the prior, average $p_\theta(x \mid z)$ — is hopeless in practice: for a given image $x$ almost every prior sample decodes to a completely different image, so almost all terms are astronomically small and the estimate is dominated by the rare lucky sample. For the same reason the **posterior**

$$
p_\theta(z \mid x) = \frac{p_\theta(x \mid z)\, p(z)}{p_\theta(x)},
$$

the distribution over codes that could have produced $x$, is intractable: its normalizer is the very integral we cannot compute.

### 4.2 Variational inference and the evidence lower bound

**Variational inference** sidesteps the intractable posterior by *approximating* it with a tractable family. We introduce a second network, the encoder or **inference network** $q_\phi(z \mid x)$, that outputs the parameters of a diagonal Gaussian for each input:

$$
q_\phi(z \mid x) = \mathcal{N}\big(z;\, \mu_\phi(x),\, \mathrm{diag}(\sigma_\phi^2(x))\big).
$$

Now take the log-likelihood and insert $q_\phi$ into the integral by multiplying and dividing by it:

$$
\log p_\theta(x)
= \log \int q_\phi(z \mid x)\, \frac{p_\theta(x \mid z)\, p(z)}{q_\phi(z \mid x)}\, dz
= \log \mathbb{E}_{z \sim q_\phi(z \mid x)} \left[ \frac{p_\theta(x \mid z)\, p(z)}{q_\phi(z \mid x)} \right].
$$

The logarithm is concave, so by **Jensen's inequality** $\log \mathbb{E}[Y] \ge \mathbb{E}[\log Y]$ for any positive random variable $Y$. Applying it,

$$
\log p_\theta(x) \;\ge\; \mathbb{E}_{q_\phi(z \mid x)} \left[ \log p_\theta(x \mid z) + \log p(z) - \log q_\phi(z \mid x) \right]
\;=\; \underbrace{\mathbb{E}_{q_\phi(z \mid x)}\big[\log p_\theta(x \mid z)\big]}_{\text{expected reconstruction log-likelihood}}
\;-\; \underbrace{\mathrm{KL}\big(q_\phi(z \mid x)\,\|\,p(z)\big)}_{\text{regularizer}}
\;=:\; \mathcal{L}(\theta, \phi; x),
$$

where we used the definition $\mathrm{KL}(q \| p) = \mathbb{E}_q[\log q - \log p]$. The quantity $\mathcal{L}$ is the **evidence lower bound** (ELBO; "evidence" is a traditional name for $\log p_\theta(x)$). Its two terms have an immediate reading: the first rewards decoders that reconstruct $x$ well from codes drawn from the encoder's cloud, the second pulls each cloud towards the prior. This is exactly the fix promised at the end of Section 3.

The Jensen derivation tells us the bound holds but not how tight it is. A second derivation does. Start from the KL divergence between the approximate and the true posterior and expand it using Bayes' rule, $\log p_\theta(z \mid x) = \log p_\theta(x \mid z) + \log p(z) - \log p_\theta(x)$:

$$
\mathrm{KL}\big(q_\phi(z \mid x)\,\|\,p_\theta(z \mid x)\big)
= \mathbb{E}_{q_\phi}\big[\log q_\phi(z \mid x) - \log p_\theta(x \mid z) - \log p(z)\big] + \log p_\theta(x)
= -\mathcal{L}(\theta, \phi; x) + \log p_\theta(x),
$$

where $\log p_\theta(x)$ came out of the expectation because it does not depend on $z$. Rearranging gives the central identity of the VAE:

$$
\boxed{\;\log p_\theta(x) = \mathcal{L}(\theta, \phi; x) + \mathrm{KL}\big(q_\phi(z \mid x)\,\|\,p_\theta(z \mid x)\big)\;}
$$

Since a KL divergence is non-negative, this re-proves $\mathcal{L} \le \log p_\theta(x)$, and it shows that the **gap** between the bound and the true log-likelihood is precisely how far the encoder's approximation is from the true posterior. Maximizing the ELBO with respect to $\phi$ therefore performs approximate posterior inference (it shrinks the gap), and maximizing with respect to $\theta$ pushes up a lower bound on the likelihood. Both happen simultaneously by gradient ascent on one objective, and the encoder is *amortized*: one network produces the approximate posterior for every $x$ in a single forward pass, instead of running a separate optimization per data point as classical variational inference would.

> **Key idea.** The ELBO turns an intractable integral into an expectation under a distribution we control, plus a KL penalty we can compute in closed form. The price is a bound instead of the exact likelihood, and the size of the gap is the posterior approximation error.

### 4.3 The reparameterization trick

To train by gradient descent we need $\nabla_\theta \mathcal{L}$ and $\nabla_\phi \mathcal{L}$. The gradient with respect to the decoder parameters $\theta$ is easy: the expectation is over $q_\phi$, which does not involve $\theta$, so we can move the gradient inside and estimate it with a single sample $z \sim q_\phi(z \mid x)$. The gradient with respect to the encoder parameters $\phi$ is the problem, because $\phi$ determines the *distribution we sample from*: $\nabla_\phi \mathbb{E}_{z \sim q_\phi}[f(z)] \neq \mathbb{E}_{z \sim q_\phi}[\nabla_\phi f(z)]$, and the right-hand side is not even meaningful, since $f(z) = \log p_\theta(x \mid z)$ does not depend on $\phi$ once $z$ is fixed.

One general answer is the **score-function estimator** (also called REINFORCE, which reappears in Notebook 13 · Reinforcement Learning): using $\nabla_\phi q_\phi = q_\phi \nabla_\phi \log q_\phi$,

$$
\nabla_\phi \mathbb{E}_{q_\phi}[f(z)] = \mathbb{E}_{q_\phi}\big[f(z)\, \nabla_\phi \log q_\phi(z \mid x)\big].
$$

It is unbiased and works for any distribution, even discrete ones, but its variance is notoriously large: the estimate multiplies the *value* $f(z)$ (a large number for a 784-pixel log-likelihood) by a score that changes sign randomly.

The **reparameterization trick** exploits the fact that a Gaussian sample can be written as a deterministic, differentiable function of the parameters and of parameter-free noise:

$$
z = \mu_\phi(x) + \sigma_\phi(x) \odot \epsilon, \qquad \epsilon \sim \mathcal{N}(0, I).
$$

The randomness now lives in $\epsilon$, whose distribution does not depend on $\phi$, so the gradient can move inside the expectation:

$$
\nabla_\phi \mathbb{E}_{z \sim q_\phi}[f(z)] = \nabla_\phi \mathbb{E}_{\epsilon}\big[f(\mu_\phi(x) + \sigma_\phi(x) \odot \epsilon)\big] = \mathbb{E}_{\epsilon}\big[\nabla_\phi f(\mu_\phi(x) + \sigma_\phi(x) \odot \epsilon)\big].
$$

The right-hand side is an ordinary backpropagation through the decoder, through the sampling formula, into the encoder. Autograd does it for free once `z = mu + std * torch.randn_like(std)` appears in the forward pass. The estimator uses the *derivative* of $f$ rather than its value, which is why its variance is far lower. The next cell measures both estimators on a toy objective where the exact gradient is known: $f(z) = z^2$ with $z \sim \mathcal{N}(\mu, 1)$, whose expectation is $\mu^2 + 1$ with derivative $2\mu$.

In [ ]:
torch.manual_seed(SEED)
mu_true = 1.0
n_samples = 20_000
eps = torch.randn(n_samples)
z = mu_true + eps                                        # reparameterized samples from N(mu, 1)

grad_reparam = 2 * z                                     # d/dmu (mu + eps)^2 = 2 (mu + eps)
grad_score = z ** 2 * (z - mu_true)                      # f(z) * d/dmu log N(z; mu, 1) = z^2 (z - mu)

print(f"exact gradient d/dmu E[z^2] = 2 mu = {2 * mu_true:.3f}")
print(f"reparameterization estimator: mean {grad_reparam.mean():.3f}, "
      f"std of a single-sample estimate {grad_reparam.std():.3f}")
print(f"score-function estimator    : mean {grad_score.mean():.3f}, "
      f"std of a single-sample estimate {grad_score.std():.3f}")

# The same reparameterization gradient obtained by autograd, as the VAE will do it:
mu_param = torch.tensor(mu_true, requires_grad=True)
z_auto = mu_param + eps[:1000]
(z_auto ** 2).mean().backward()
print(f"autograd through z = mu + eps on 1000 samples: {mu_param.grad.item():.3f}")

Both estimators are unbiased (their means agree with $2\mu = 2$), but the single-sample standard deviation of the score-function estimator is almost three times larger — a seven-fold difference in variance, on the *easiest* possible objective. For a decoder log-likelihood that spans hundreds of nats the gap is far more dramatic. The reparameterization trick is what made the VAE trainable with ordinary stochastic gradient descent. Its limitation is that it needs a differentiable sampling formula, which discrete latent variables do not have; the VQ-VAE in Section 8.1 and Exercise 5 uses a different device for that case.

### 4.4 The KL term in closed form

The KL divergence between the encoder's diagonal Gaussian and the standard normal prior needs no sampling at all. Consider one latent dimension with $q(z) = \mathcal{N}(\mu, \sigma^2)$ and $p(z) = \mathcal{N}(0, 1)$. Writing out both log-densities,

$$
\log q(z) = -\tfrac{1}{2}\log(2\pi\sigma^2) - \frac{(z - \mu)^2}{2\sigma^2}, \qquad
\log p(z) = -\tfrac{1}{2}\log(2\pi) - \frac{z^2}{2},
$$

and taking the expectation of their difference under $q$,

$$
\mathrm{KL}(q \,\|\, p) = \mathbb{E}_q\big[\log q(z) - \log p(z)\big]
= -\tfrac{1}{2}\log \sigma^2 - \frac{\mathbb{E}_q[(z-\mu)^2]}{2\sigma^2} + \frac{\mathbb{E}_q[z^2]}{2}.
$$

Under $q$ we have $\mathbb{E}_q[(z - \mu)^2] = \sigma^2$ (the definition of the variance) and $\mathbb{E}_q[z^2] = \mathrm{Var}(z) + (\mathbb{E}\,z)^2 = \sigma^2 + \mu^2$. Substituting,

$$
\mathrm{KL}(q \,\|\, p) = -\tfrac{1}{2}\log\sigma^2 - \tfrac{1}{2} + \tfrac{1}{2}(\sigma^2 + \mu^2)
= \tfrac{1}{2}\big(\mu^2 + \sigma^2 - 1 - \log \sigma^2\big).
$$

A diagonal Gaussian factorizes over dimensions, and the KL of a product is the sum of the KLs, so for a $d$-dimensional code

$$
\boxed{\;\mathrm{KL}\big(\mathcal{N}(\mu, \mathrm{diag}\,\sigma^2)\,\|\,\mathcal{N}(0, I)\big) = \frac{1}{2}\sum_{j=1}^{d}\big(\mu_j^2 + \sigma_j^2 - 1 - \log\sigma_j^2\big).\;}
$$

Each summand is zero exactly when $\mu_j = 0$ and $\sigma_j = 1$, and positive otherwise; it grows quadratically in $\mu_j$ and penalizes both very small variances (through $-\log\sigma_j^2$) and very large ones (through $\sigma_j^2$). In code the encoder outputs the **log-variance** $\ell_j = \log \sigma_j^2$ rather than $\sigma_j$, so that any real number is a valid output and $\sigma_j = \exp(\ell_j / 2)$ is automatically positive. The next cell checks the formula against a Monte Carlo estimate and against `torch.distributions`.

In [ ]:
torch.manual_seed(SEED)
mu_q = torch.tensor([0.5, -1.0, 2.0])
logvar_q = torch.tensor([0.0, -1.0, 0.5])
kl_closed = 0.5 * (mu_q ** 2 + logvar_q.exp() - 1.0 - logvar_q)

q = torch.distributions.Normal(mu_q, torch.exp(0.5 * logvar_q))
p = torch.distributions.Normal(torch.zeros(3), torch.ones(3))
z_mc = q.sample((200_000,))                                     # (200000, 3)
kl_mc = (q.log_prob(z_mc) - p.log_prob(z_mc)).mean(dim=0)
kl_lib = torch.distributions.kl_divergence(q, p)

for j in range(3):
    print(f"dim {j}: mu={mu_q[j]:5.2f} logvar={logvar_q[j]:5.2f} | closed form {kl_closed[j]:.4f} "
          f"| Monte Carlo {kl_mc[j]:.4f} | torch.distributions {kl_lib[j]:.4f}")

The three columns agree to Monte Carlo precision. Note that dimension 2, with $\mu = 2$, costs the most: pushing a code far from the origin is expensive, and the encoder will only do it when the reconstruction gain justifies the price.

### 4.5 The decoder distribution: Bernoulli versus Gaussian

The reconstruction term $\mathbb{E}_{q}[\log p_\theta(x \mid z)]$ requires a choice of output distribution, and this choice *is* the choice of reconstruction loss.

**Bernoulli decoder.** If each pixel $x_i \in \{0, 1\}$ is modelled as an independent Bernoulli variable with probability $\hat{x}_i = \sigma(\text{logit}_i(z))$, then

$$
\log p_\theta(x \mid z) = \sum_{i=1}^{784} \big[x_i \log \hat{x}_i + (1 - x_i) \log (1 - \hat{x}_i)\big] = -\,\mathrm{BCE}(x, \hat{x}),
$$

the negative binary cross-entropy summed over pixels. MNIST pixels are grey levels in $[0, 1]$ rather than exact zeros and ones, so using this formula with real-valued targets is, strictly speaking, an abuse (the normalization of the distribution is off, and the continuous Bernoulli of Loaiza-Ganem and Cunningham, 2019, repairs it), but it is universal practice, and for the nearly binary MNIST images it works well. The numbers it produces are in **nats per image**.

**Gaussian decoder.** If instead $p_\theta(x \mid z) = \mathcal{N}(x;\, \hat{x}(z), \sigma^2 I)$ with a fixed observation noise $\sigma$,

$$
\log p_\theta(x \mid z) = -\frac{1}{2\sigma^2}\|x - \hat{x}(z)\|^2 - \frac{D}{2}\log(2\pi\sigma^2),
$$

which is the squared error up to a constant and a scale. The scale is not innocent: the reconstruction term is weighted by $1/(2\sigma^2)$ relative to the KL term, so a fixed $\sigma$ silently acts like a $\beta$ weight on the KL (Section 5.7). Choosing $\sigma^2 = 1$ and "MSE plus KL" — a common beginner's recipe — corresponds to an absurdly noisy observation model for pixels in $[0, 1]$, and results in blurry samples and a KL term that dominates. Learning $\sigma$ (Exercise 3) or using the Bernoulli likelihood removes this arbitrariness. We use the Bernoulli likelihood in this notebook.

### 4.6 The full objective

Putting the pieces together, for one image $x$ and a single reparameterized sample $\epsilon$,

$$
-\mathcal{L}(\theta, \phi; x) \;\approx\; \underbrace{\mathrm{BCE}\big(x,\; \sigma(\mathrm{dec}_\theta(\mu_\phi(x) + \sigma_\phi(x) \odot \epsilon))\big)}_{\text{reconstruction NLL}}
\;+\; \underbrace{\frac{1}{2}\sum_{j}\big(\mu_j^2 + \sigma_j^2 - 1 - \log\sigma_j^2\big)}_{\text{KL to the prior}},
$$

and training minimizes the mini-batch average of this **negative ELBO** with Adam. A single sample of $\epsilon$ per image per step suffices — the mini-batch already averages over many images, and more samples per image rarely help. Both terms are in nats, so their sum is a lower bound on $-\log p_\theta(x)$ and can be compared across models: a good MNIST VAE reaches a negative ELBO around 90–100 nats on the test set, a mediocre one 130 and above.

> **Common pitfall.** Averaging the BCE over pixels (`reduction="mean"`) while summing the KL over latent dimensions divides the reconstruction term by 784 relative to the KL. The KL then dominates, the encoder collapses onto the prior, and every reconstruction is the same blurry mean digit. Always use per-image *sums* for both terms, or scale them consistently.

## 5. A VAE on MNIST

### 5.1 Implementation

The implementation reuses the convolutional encoder and decoder from Section 2. The only changes are that the encoder now outputs $2d$ numbers — the means and the log-variances of the $d$ latent dimensions — and that a reparameterized sample, rather than a deterministic code, is fed to the decoder. The loss function returns the reconstruction and KL terms separately so that we can watch them individually during training; the `beta` argument, equal to 1 for the plain VAE, will be used for KL annealing and the $\beta$-VAE later.

In [ ]:
class ConvVAE(nn.Module):
    """Convolutional VAE with a diagonal-Gaussian encoder and a Bernoulli (logit) decoder."""

    def __init__(self, latent_dim, width=WIDTH):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = ConvEncoder(2 * latent_dim, width)
        self.decoder = ConvDecoder(latent_dim, width)

    def encode(self, x):
        mu, logvar = self.encoder(x).chunk(2, dim=1)
        return mu, logvar

    @staticmethod
    def reparameterize(mu, logvar):
        eps = torch.randn_like(mu)
        return mu + eps * torch.exp(0.5 * logvar)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def kl_standard_normal(mu, logvar):
    """KL( N(mu, diag(exp(logvar))) || N(0, I) ), summed over latent dimensions: one value per example."""
    return 0.5 * (mu.pow(2) + logvar.exp() - 1.0 - logvar).sum(dim=1)


@torch.no_grad()
def evaluate_vae(model, X, Y=None, rec_fn=recon_bce, batch_size=1000):
    """Mean reconstruction NLL, KL and negative ELBO (nats per image) on a tensor of images."""
    model.eval()
    rec_sum = kl_sum = 0.0
    tensors = [X] if Y is None else [X, Y]
    for batch in iterate_batches(tensors, batch_size, shuffle=False):
        logits, mu, logvar = model(*batch)
        rec_sum += rec_fn(logits, batch[0]).sum().item()
        kl_sum += kl_standard_normal(mu, logvar).sum().item()
    n = X.shape[0]
    return rec_sum / n, kl_sum / n, (rec_sum + kl_sum) / n


def train_vae(model, X, epochs, Y=None, lr=1e-3, batch_size=BATCH_SIZE, beta=1.0,
              beta_schedule=None, rec_fn=recon_bce, verbose=True):
    """Train a VAE (or a conditional VAE when labels Y are given) by minimizing the negative ELBO.

    beta_schedule(step, total_steps) -> beta, if given, overrides the constant beta (KL annealing).
    rec_fn(logits, x) -> per-image reconstruction NLL (default: Bernoulli / BCE).
    Returns a history with per-step reconstruction, KL and beta values and per-epoch test metrics.
    """
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    tensors = [X] if Y is None else [X, Y]
    steps_per_epoch = math.ceil(X.shape[0] / batch_size)
    total_steps, step = epochs * steps_per_epoch, 0
    hist = {"rec": [], "kl": [], "beta": [], "test_rec": [], "test_kl": [], "test_nelbo": []}
    for epoch in range(1, epochs + 1):
        model.train(); t0 = time.time()
        for batch in iterate_batches(tensors, batch_size):
            b = beta if beta_schedule is None else beta_schedule(step, total_steps)
            logits, mu, logvar = model(*batch)
            rec = rec_fn(logits, batch[0]).mean()
            kl = kl_standard_normal(mu, logvar).mean()
            loss = rec + b * kl
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            hist["rec"].append(rec.item()); hist["kl"].append(kl.item()); hist["beta"].append(b)
            step += 1
        test_rec, test_kl, test_nelbo = evaluate_vae(model, X_test, None if Y is None else y_test, rec_fn)
        hist["test_rec"].append(test_rec); hist["test_kl"].append(test_kl); hist["test_nelbo"].append(test_nelbo)
        if verbose:
            print(f"epoch {epoch:2d} | train rec {np.mean(hist['rec'][-steps_per_epoch:]):7.2f} "
                  f"kl {np.mean(hist['kl'][-steps_per_epoch:]):6.2f} | test rec {test_rec:7.2f} "
                  f"kl {test_kl:6.2f} -ELBO {test_nelbo:7.2f} | beta {b:.2f} | {time.time() - t0:.1f}s")
    return hist

### 5.2 Training and the two loss curves

We train two models: one with $d = 2$, whose latent plane we can draw exhaustively, and one with $d = 16$, which reconstructs and generates much better and serves as the reference model for the rest of the notebook. Watch the two terms in the printed lines: the reconstruction term starts around the value of a "mean image" decoder and falls, while the KL term starts near zero (an untrained encoder outputs $\mu \approx 0$, $\log\sigma^2 \approx 0$, which is the prior) and *rises* as the encoder learns that spending nats on informative codes pays off in reconstruction.

In [ ]:
torch.manual_seed(SEED)
vae2 = ConvVAE(2).to(device)
print("--- VAE with latent dimension 2 ---")
hist2 = train_vae(vae2, X_train, EPOCHS_VAE)

torch.manual_seed(SEED)
vae16 = ConvVAE(16).to(device)
print("--- VAE with latent dimension 16 ---")
hist16 = train_vae(vae16, X_train, EPOCHS_VAE)

The per-epoch summaries hide the dynamics. Plotting the two terms *per mini-batch step* shows how the reconstruction term and the KL term evolve together, and how the two latent sizes differ in the amount of information they end up transmitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for hist, lab in [(hist2, "d = 2"), (hist16, "d = 16")]:
    axes[0].plot(hist["rec"], label=lab, alpha=0.8)
    axes[1].plot(hist["kl"], label=lab, alpha=0.8)
axes[0].set_title("reconstruction NLL per image (train, per step)"); axes[0].set_xlabel("step")
axes[0].set_ylabel("nats"); axes[0].legend()
axes[1].set_title("KL( q(z|x) || p(z) ) per image (train, per step)"); axes[1].set_xlabel("step")
axes[1].set_ylabel("nats"); axes[1].legend()
plt.tight_layout(); plt.show()

for d, hist in [(2, hist2), (16, hist16)]:
    print(f"d={d:2d}: final test reconstruction {hist['test_rec'][-1]:7.2f} nats | KL {hist['test_kl'][-1]:6.2f} nats "
          f"| -ELBO {hist['test_nelbo'][-1]:7.2f} nats")
print(f"for comparison, the deterministic AE with d=32 reaches a test BCE of {ae_hist[32]['test_bce'][-1]:.2f} nats")

Three observations. First, the $d = 16$ model reaches a much lower reconstruction term than the $d = 2$ model, while its KL term is larger: more latent dimensions let the encoder transmit more information about the image, and every nat transmitted is paid for in the KL term. Second, the KL of the 2-dimensional model saturates early — two Gaussian coordinates can only carry so much information about a digit, and the encoder quickly uses all of it. Third, the VAE's reconstruction term is worse than the deterministic autoencoder's, as it must be: the decoder has to reconstruct from a *noisy* code, and the KL penalty keeps the noise from shrinking to zero. That loss in fidelity buys a latent space with a known prior and a smooth decoder, which is exactly what we need for generation.

### 5.3 Sampling from the prior

Generation is now a two-liner: draw $z \sim \mathcal{N}(0, I)$, decode, and take the sigmoid to obtain pixel probabilities (showing the probabilities rather than Bernoulli samples is the usual convention).

In [ ]:
@torch.no_grad()
def sample_vae(model, n, seed=SEED):
    """Decode n latent vectors drawn from the standard normal prior."""
    model.eval()
    torch.manual_seed(seed)
    z = torch.randn(n, model.latent_dim, device=device)
    return torch.sigmoid(model.decode(z))

show_images(sample_vae(vae16, 40), nrow=10, title="VAE (d = 16): samples from the prior", figsize=(7, 3.2))
show_images(sample_vae(vae2, 40), nrow=10, title="VAE (d = 2): samples from the prior", figsize=(7, 3.2))

Compare these with the autoencoder samples in Section 3: nearly every VAE sample is a plausible digit, because every prior sample lands in a region the decoder was trained on. Under the `FAST_RUN` budget (a couple of epochs on a subset) the digits are soft and some are ambiguous; with the full training budget they become crisp. Either way the characteristic **blurriness** of VAE samples is visible: the model spreads probability over several nearby digit shapes and the per-pixel Bernoulli decoder averages them. Section 7 quantifies sample quality, and Notebook 12 · GANs and Diffusion Models introduces model families that trade the likelihood bound for sharper samples.

### 5.4 The two-dimensional latent manifold

With $d = 2$ we can render the entire generative model as one picture. We lay a grid over the latent plane, decode every grid point, and tile the results. The grid is placed at the quantiles of the standard normal rather than at evenly spaced coordinates: this samples the plane in proportion to the prior, so that the picture shows what the model actually generates rather than the improbable far tails.

In [ ]:
@torch.no_grad()
def latent_manifold(model, n=15, span=0.98):
    """Decode an n x n grid of latent points placed at quantiles of the N(0, 1) prior (d = 2 only)."""
    model.eval()
    q = torch.linspace(1.0 - span, span, n)
    grid_vals = torch.distributions.Normal(0.0, 1.0).icdf(q).to(device)
    zz = torch.stack(torch.meshgrid(grid_vals, grid_vals, indexing="ij"), dim=-1).reshape(-1, 2)
    imgs = torch.sigmoid(model.decode(zz)).reshape(n, n, 28, 28)          # imgs[i, j] at (z1_i, z2_j)
    canvas = imgs.permute(1, 0, 2, 3).flip(0)                              # rows: z2 descending; cols: z1 ascending
    return canvas.permute(0, 2, 1, 3).reshape(n * 28, n * 28).cpu().numpy(), grid_vals.cpu().numpy()

@torch.no_grad()
def encode_vae(model, X, batch_size=1000):
    """Posterior means and log-variances of a VAE encoder for a tensor of images."""
    model.eval()
    mus, logvars = [], []
    for (xb,) in iterate_batches([X], batch_size, shuffle=False):
        mu, logvar = model.encode(xb)
        mus.append(mu); logvars.append(logvar)
    return torch.cat(mus), torch.cat(logvars)

canvas, grid_vals = latent_manifold(vae2)
mu2_test, _ = encode_vae(vae2, X_test)

fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
axes[0].imshow(canvas, cmap="gray", vmin=0, vmax=1,
               extent=[grid_vals[0], grid_vals[-1], grid_vals[0], grid_vals[-1]])
axes[0].set_xlabel("$z_1$"); axes[0].set_ylabel("$z_2$"); axes[0].grid(False)
axes[0].set_title("VAE (d = 2): decoded latent grid")

mu2_np = mu2_test.cpu().numpy()
sc = axes[1].scatter(mu2_np[:, 0], mu2_np[:, 1], c=y_test_np, cmap="tab10", s=3, alpha=0.6)
axes[1].set_xlim(grid_vals[0], grid_vals[-1]); axes[1].set_ylim(grid_vals[0], grid_vals[-1])
axes[1].set_xlabel("$z_1$"); axes[1].set_ylabel("$z_2$"); axes[1].set_title("posterior means of the test set")
plt.colorbar(sc, ax=axes[1], ticks=range(10), label="digit")
plt.tight_layout(); plt.show()

The left panel is the decoder's whole world: digits morph continuously into their neighbours, and there are no garbage regions, because the KL term has pulled the encoder's clouds together into a blob that fills the prior's high-density region. The right panel confirms it from the encoder side. Compare with the autoencoder scatter of Section 2.4: the codes are now centred at the origin with unit scale, the clusters touch, and the gaps are gone. The price, visible in both panels, is that neighbouring classes overlap — a 2-dimensional Gaussian bottleneck cannot separate ten classes cleanly *and* keep the space filled.

### 5.5 Interpolation

Interpolating between two test images in the VAE's latent space uses the posterior means. Because the whole latent region is now covered, straight lines stay in well-trained territory, and the intermediate images are cleaner than those of the autoencoder in Section 2.4.

In [ ]:
@torch.no_grad()
def interpolate_vae(model, xa, xb, steps=10):
    """Decode the straight line between the posterior means of two images."""
    model.eval()
    za, _ = model.encode(xa[None]); zb, _ = model.encode(xb[None])
    t = torch.linspace(0, 1, steps, device=za.device)[:, None]
    return torch.sigmoid(model.decode((1 - t) * za + t * zb))

show_rows([pixel_path, interpolate_ae(aes[32], xa, xb), interpolate_vae(vae16, xa, xb)],
          ["pixel space", "AE, d=32", "VAE, d=16"], figsize=(7, 3))

> **Going deeper.** In high dimensions the straight line between two prior samples passes through a region of *low* prior density: the norm of a $d$-dimensional standard normal sample concentrates around $\sqrt{d}$, but the midpoint of two such samples has norm around $\sqrt{d/2}$. Spherical interpolation (`slerp`), which keeps the norm constant, is the better choice for interpolating between *prior samples*; for interpolating between *posterior means* of real images, as here, linear interpolation is usually fine.

### 5.6 Posterior collapse and KL annealing

The KL term is a double-edged sword. At the start of training the decoder is useless, so the reconstruction term gives the encoder no reason to transmit information, while the KL term punishes every nat it transmits. The easiest way to reduce the loss is to set $q_\phi(z \mid x) = p(z)$ for every $x$: KL exactly zero, and the decoder learns to output the average digit regardless of $z$. This is **posterior collapse**. Once collapsed, the model can be stuck: with no information in $z$, the decoder never learns to use it, so the encoder never gets a gradient telling it to send any. Collapse is most severe when the decoder is powerful enough to model the data *without* $z$ — autoregressive decoders on text are the classic case — but it is also common in milder forms, where only a subset of the latent dimensions is used and the rest sit exactly at the prior. Those unused dimensions are called **inactive units**.

The standard first remedy is **KL annealing** (Bowman et al., 2016): multiply the KL term by a weight $\beta_t$ that starts at 0 and increases linearly to 1 over the first part of training. Early on the model is a plain autoencoder and learns to put information in $z$; by the time the penalty reaches full strength, the decoder depends on $z$ and the information is worth keeping. Other remedies include *free bits* (do not penalize the first few nats of KL per dimension), weakening the decoder (dropout on its inputs) or the more principled $\delta$-VAE constraints. We train a second $d = 16$ model with a linear warm-up over the first half of training and compare the KL trajectories.

In [ ]:
def linear_warmup(step, total_steps, warm_frac=0.5):
    """KL weight rising linearly from 0 to 1 over the first warm_frac of training."""
    return min(1.0, step / max(1.0, warm_frac * total_steps))

torch.manual_seed(SEED)
vae16_anneal = ConvVAE(16).to(device)
print("--- VAE with latent dimension 16, linear KL annealing over the first half of training ---")
hist16_anneal = train_vae(vae16_anneal, X_train, EPOCHS_VAE, beta_schedule=linear_warmup)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(hist16["kl"], label="constant beta = 1", alpha=0.8)
axes[0].plot(hist16_anneal["kl"], label="annealed beta", alpha=0.8)
axes[0].set_title("KL term per step"); axes[0].set_xlabel("step"); axes[0].set_ylabel("nats"); axes[0].legend()
axes[1].plot(hist16["rec"], label="constant beta = 1", alpha=0.8)
axes[1].plot(hist16_anneal["rec"], label="annealed beta", alpha=0.8)
ax_b = axes[1].twinx(); ax_b.plot(hist16_anneal["beta"], color="gray", ls=":", label="beta (annealed run)")
ax_b.set_ylabel("beta"); ax_b.grid(False)
axes[1].set_title("reconstruction term per step"); axes[1].set_xlabel("step"); axes[1].set_ylabel("nats")
axes[1].legend(loc="upper right"); ax_b.legend(loc="center right")
plt.tight_layout(); plt.show()

During the warm-up the annealed model behaves like an autoencoder: its KL shoots far above the constant-$\beta$ run's (it transmits freely) and its reconstruction term falls faster. As $\beta$ approaches 1 the KL is squeezed back down and the two runs converge towards similar values — on this small convolutional model there is no severe collapse to rescue, so annealing mostly changes the *path*, not the destination. Where it matters is in how many latent dimensions end up carrying information. A standard diagnostic (Burda et al., 2016) calls a latent unit **active** if the variance of its posterior mean across the data exceeds 0.01; an inactive unit outputs the same $\mu_j \approx 0$ for every image. We also report the average KL contributed by each dimension.

In [ ]:
@torch.no_grad()
def latent_usage(model, X):
    """Per-dimension mean KL (nats) and the number of active units (Var_x[mu_j(x)] > 0.01)."""
    mu, logvar = encode_vae(model, X)
    kl_per_dim = (0.5 * (mu.pow(2) + logvar.exp() - 1.0 - logvar)).mean(dim=0)
    active = (mu.var(dim=0) > 0.01).sum().item()
    return kl_per_dim.cpu().numpy(), active

kl_dims_const, active_const = latent_usage(vae16, X_test)
kl_dims_anneal, active_anneal = latent_usage(vae16_anneal, X_test)
order = np.argsort(-kl_dims_const)
x_pos = np.arange(16)
plt.bar(x_pos - 0.2, kl_dims_const[order], width=0.4, label=f"constant beta ({active_const} active units)")
plt.bar(x_pos + 0.2, kl_dims_anneal[order], width=0.4, label=f"annealed ({active_anneal} active units)")
plt.xlabel("latent dimension (sorted by KL of the constant-beta model)"); plt.ylabel("mean KL (nats)")
plt.title("How much information each latent dimension carries"); plt.legend()
plt.tight_layout(); plt.show()

The KL is spread unevenly: a few dimensions carry several nats each while others contribute almost nothing. Dimensions with (nearly) zero KL are the collapsed ones. Whether annealing rescues additional units depends on the budget and the seed; what is robust is the shape of the profile, which tells you the *effective* latent dimensionality of the model — often noticeably smaller than $d$. Exercise 1 sweeps $d$ and shows that the number of active units saturates.

### 5.7 The beta-VAE and the rate–distortion view

Higgins et al. (2017) proposed to weight the KL term by a constant $\beta$:

$$
-\mathcal{L}_\beta = \mathrm{NLL}_{\text{rec}} + \beta\, \mathrm{KL}\big(q_\phi(z \mid x)\,\|\,p(z)\big).
$$

With $\beta > 1$ the encoder is pushed harder towards the prior, so it transmits fewer nats: reconstructions blur, but the latent dimensions that survive tend to be more *disentangled* — each aligned with one interpretable factor of variation such as slant or thickness — because the cheapest way to convey information under a strong independent-Gaussian prior is with independent coordinates. With $\beta < 1$ the model is closer to a plain autoencoder: sharper reconstructions, a less regular latent space.

The cleanest way to think about $\beta$ is **rate–distortion** (Alemi et al., 2018). The KL term is the **rate** $R$: the number of nats the encoder sends about $x$. The reconstruction term is the **distortion** $D$. Every model traces a point $(R, D)$, the ELBO is $R + D$, and varying $\beta$ walks along the *rate–distortion curve* of the architecture: $\beta$ is the slope of the tangent, i.e. the exchange rate between a nat of rate and a nat of distortion. There is no single "right" $\beta$; there is a curve, and the application decides where on it to sit. The sweep below trains $\beta = 0.25$ and $\beta = 4$ and reuses the $\beta = 1$ model.

In [ ]:
BETAS = [0.25, 1.0, 4.0]
vaes_beta, hist_beta = {1.0: vae16}, {1.0: hist16}
for beta in BETAS:
    if beta in vaes_beta:
        continue
    torch.manual_seed(SEED)
    print(f"--- beta-VAE with latent dimension 16, beta = {beta} ---")
    vaes_beta[beta] = ConvVAE(16).to(device)
    hist_beta[beta] = train_vae(vaes_beta[beta], X_train, EPOCHS_VAE, beta=beta)

print("\nbeta   rate (KL)  distortion (rec NLL)  -ELBO (rec + KL)  active units")
rd_points = {}
for beta in BETAS:
    rec, kl, nelbo = evaluate_vae(vaes_beta[beta], X_test)
    _, active = latent_usage(vaes_beta[beta], X_test)
    rd_points[beta] = (kl, rec)
    print(f"{beta:4.2f}   {kl:8.2f}   {rec:16.2f}   {nelbo:14.2f}   {active:8d}")

plt.figure(figsize=(5.5, 4))
for beta, (kl, rec) in rd_points.items():
    plt.scatter(kl, rec, s=60); plt.annotate(f"beta = {beta}", (kl, rec), textcoords="offset points", xytext=(6, 4))
plt.xlabel("rate: KL (nats per image)"); plt.ylabel("distortion: reconstruction NLL (nats per image)")
plt.title("Rate-distortion points of the beta sweep")
plt.tight_layout(); plt.show()

for beta in BETAS:
    show_images(sample_vae(vaes_beta[beta], 20), nrow=10, title=f"prior samples, beta = {beta}", figsize=(7, 1.9))

The table reads exactly as the theory predicts: as $\beta$ grows the rate falls and the distortion rises, and the number of active units shrinks. The $\beta = 1$ model should have the best (lowest) negative ELBO, since it is the only one that optimizes that quantity directly (with a short training budget the margin over $\beta = 0.25$ can be small, because both models are still improving). The samples tell the complementary story: the low-$\beta$ model reconstructs best but its prior samples are the least digit-like (its aggregate posterior has drifted from the prior, so prior samples land in under-trained regions — the autoencoder problem creeping back), whereas the high-$\beta$ model produces the most generic, blurriest, but consistently digit-shaped samples.

> **Try it.** Set `BETAS = [0.1, 1.0, 10.0]` and rerun. At $\beta = 10$ on this small model you should see the KL drop towards zero and the samples become nearly identical — full posterior collapse, on purpose.

## 6. Conditional VAE

So far the model decides *which* digit to draw by itself, through the latent code. Often we want to control that — "draw a 7" — while keeping the randomness for the style. The **conditional VAE** (Sohn, Lee and Yan, 2015) conditions both networks on side information $y$, here the class label:

$$
\mathcal{L}(\theta, \phi; x, y) = \mathbb{E}_{q_\phi(z \mid x, y)}\big[\log p_\theta(x \mid z, y)\big] - \mathrm{KL}\big(q_\phi(z \mid x, y)\,\|\,p(z)\big).
$$

The derivation of Section 4 goes through unchanged with every distribution conditioned on $y$; the prior may also depend on $y$ but we keep it standard normal. Because the label already tells the decoder which digit to draw, the latent code no longer needs to encode identity, only style — and the KL term should shrink accordingly, by roughly the $\log 10 \approx 2.3$ nats that the class identity is worth.

Implementation-wise, the encoder receives the label as ten extra constant input channels (a one-hot vector broadcast over the image), and the decoder receives the one-hot vector concatenated to $z$. Both are the simplest possible conditioning mechanisms; Notebook 12 · GANs and Diffusion Models uses learned label embeddings added to hidden features instead, which scales better.

In [ ]:
class ConvCVAE(nn.Module):
    """Conditional VAE: the encoder sees (x, y) and the decoder sees (z, y); y is an integer label."""

    def __init__(self, latent_dim, n_classes=10, width=WIDTH):
        super().__init__()
        self.latent_dim, self.n_classes = latent_dim, n_classes
        self.encoder = ConvEncoder(2 * latent_dim, width, in_channels=1 + n_classes)
        self.decoder = ConvDecoder(latent_dim + n_classes, width)

    def encode(self, x, y):
        y_maps = F.one_hot(y, self.n_classes).float()[:, :, None, None].expand(-1, -1, 28, 28)
        mu, logvar = self.encoder(torch.cat([x, y_maps], dim=1)).chunk(2, dim=1)
        return mu, logvar

    def decode(self, z, y):
        return self.decoder(torch.cat([z, F.one_hot(y, self.n_classes).float()], dim=1))

    def forward(self, x, y):
        mu, logvar = self.encode(x, y)
        z = ConvVAE.reparameterize(mu, logvar)
        return self.decode(z, y), mu, logvar

torch.manual_seed(SEED)
cvae = ConvCVAE(16).to(device)
print("--- conditional VAE with latent dimension 16 ---")
hist_cvae = train_vae(cvae, X_train, EPOCHS_VAE, Y=y_train)
print(f"test KL: unconditional VAE {hist16['test_kl'][-1]:.2f} nats | conditional VAE {hist_cvae['test_kl'][-1]:.2f} nats")

The KL term of the conditional model is indeed lower than that of the unconditional one, and its reconstruction term is better too: the decoder gets the class for free, so the same latent budget goes further. To see the division of labour between $y$ and $z$ we draw a grid in which each row requests a different digit and each column uses the *same* latent vector across rows. If the model has learned what we hoped, each column should show a consistent style — slant, thickness, size — applied to ten different digits.

In [ ]:
@torch.no_grad()
def sample_cvae(model, digits, n_per_digit=8, seed=SEED):
    """Decode n_per_digit prior samples (shared across digits) for every requested digit."""
    model.eval()
    torch.manual_seed(seed)
    z = torch.randn(n_per_digit, model.latent_dim, device=device)
    rows = []
    for dgt in digits:
        y = torch.full((n_per_digit,), int(dgt), device=device, dtype=torch.long)
        rows.append(torch.sigmoid(model.decode(z, y)))
    return torch.cat(rows)

show_images(sample_cvae(cvae, range(10)), nrow=8, figsize=(5.5, 7),
            title="Conditional VAE: rows = requested digit, columns = shared latent z")

# A style transfer: encode one real image with its label, then decode its code with every label.
with torch.no_grad():
    cvae.eval()
    x_src, y_src = X_test[:1], y_test[:1]
    mu_src, _ = cvae.encode(x_src, y_src)
    restyled = torch.sigmoid(cvae.decode(mu_src.expand(10, -1), torch.arange(10, device=device)))
show_images(torch.cat([x_src, restyled]), nrow=11, figsize=(7, 1.4),
            title="Source image (left) and its latent code decoded as digits 0-9")

Reading down a column, the digit changes while the handwriting stays recognizably the same; reading across a row, the digit is fixed and the style varies. The style-transfer row makes the point with a real image: the code of one handwritten digit, decoded with the other nine labels, produces the other nine digits *in the same hand*. This is the simplest instance of a **disentangled** representation — one factor (identity) supplied explicitly, the remaining factors (style) captured by $z$ — and the same conditioning principle underlies class-conditional diffusion models, which we implement in Notebook 12 · GANs and Diffusion Models.

## 7. Evaluating generative models

How good is a generative model? The question is harder than for classifiers, because there is no single ground truth: a model that memorized the training set would reproduce it perfectly and be useless, and a model that produced beautiful digits of only one class would be sharp but not diverse. Several families of metrics exist, each capturing a different aspect.

**Likelihood-based metrics.** For a VAE the negative ELBO on held-out data is a principled score: it bounds the negative log-likelihood, so a lower value means the model assigns more probability to real data it has not seen. Exercise 4 tightens it with importance sampling. Likelihoods are comparable across VAEs but not across model families: GANs have no likelihood at all, and diffusion models have a bound of a different kind. Likelihoods are also known to correlate imperfectly with perceived sample quality — a model can score well by getting the background pixels right.

**Reconstruction metrics.** The reconstruction term alone, or the squared error, measures how well the model *compresses* rather than how well it *generates*; we used it for the autoencoders.

**Sample-based metrics.** The **Inception Score** (Salimans et al., 2016) runs an ImageNet classifier on generated samples and rewards confident predictions (fidelity) with a uniform marginal over classes (diversity). The **Fréchet Inception Distance** (FID; Heusel et al., 2017) is now the standard: embed real and generated images with the penultimate layer of an Inception network, fit a Gaussian to each set of features, and compute the Fréchet (Wasserstein-2) distance between the two Gaussians,

$$
\mathrm{FID} = \|\mu_r - \mu_g\|^2 + \mathrm{tr}\big(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2}\big).
$$

Lower is better; it captures both fidelity (the means must match) and diversity (the covariances must match). It has known weaknesses — a bias that depends on the number of samples, sensitivity to the feature extractor, and blindness to anything the classifier does not encode — and *precision / recall* style metrics (Kynkäänniemi et al., 2019) were introduced to separate fidelity from diversity explicitly.

We cannot run an Inception network on MNIST meaningfully, but the *mechanism* transfers: we train a small MNIST classifier and use its 64-dimensional penultimate layer as the feature extractor. The resulting **feature distance** is a proxy for FID. We also report the classifier's average confidence on the samples (fidelity: are they clearly some digit?) and the entropy of the predicted-class histogram (diversity: are all ten digits produced? the maximum is $\log 10 \approx 2.30$ nats).

In [ ]:
class SmallCNN(nn.Module):
    """A compact MNIST classifier whose 64-d penultimate layer serves as a feature extractor."""

    def __init__(self, width=WIDTH):
        super().__init__()
        self.features_net = nn.Sequential(
            nn.Conv2d(1, width, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),               # 28 -> 14
            nn.Conv2d(width, 2 * width, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),       # 14 -> 7
            nn.Flatten(), nn.Linear(2 * width * 7 * 7, 64), nn.ReLU(),
        )
        self.head = nn.Linear(64, 10)

    def features(self, x):
        return self.features_net(x)

    def forward(self, x):
        return self.head(self.features(x))

torch.manual_seed(SEED)
clf = SmallCNN().to(device)
opt = torch.optim.Adam(clf.parameters(), lr=1e-3)
EPOCHS_CLF = 1 if FAST_RUN else 3
for epoch in range(1, EPOCHS_CLF + 1):
    clf.train(); t0 = time.time()
    for xb, yb in iterate_batches([X_train, y_train], BATCH_SIZE):
        loss = F.cross_entropy(clf(xb), yb)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    clf.eval()
    with torch.no_grad():
        correct = sum((clf(xb).argmax(dim=1) == yb).sum().item()
                      for xb, yb in iterate_batches([X_test, y_test], 1000, shuffle=False))
    print(f"classifier epoch {epoch} | test accuracy {correct / X_test.shape[0]:.4f} | {time.time() - t0:.1f}s")

With the classifier in place, the next cell defines the feature statistics and the Fréchet distance, builds a set of image collections — real images, VAE reconstructions, samples from every model trained so far, and the "Gaussian-code" autoencoder samples from Section 3 — and scores them all against a held-out set of real test images.

In [ ]:
@torch.no_grad()
def classifier_stats(X, batch_size=1000):
    """Penultimate features (numpy, float64), mean max-softmax confidence and predicted-class entropy."""
    clf.eval()
    feats, probs = [], []
    for (xb,) in iterate_batches([X], batch_size, shuffle=False):
        f = clf.features(xb)
        feats.append(f); probs.append(F.softmax(clf.head(f), dim=1))
    feats = torch.cat(feats).cpu().numpy().astype(np.float64)
    probs = torch.cat(probs)
    confidence = probs.max(dim=1).values.mean().item()
    counts = torch.bincount(probs.argmax(dim=1), minlength=10).double()
    p = counts / counts.sum()
    entropy = -(p * torch.log(p + 1e-12)).sum().item()
    return feats, confidence, entropy

def frechet_distance(f1, f2):
    """Fréchet distance between Gaussians fitted to two feature sets (the FID formula)."""
    mu1, mu2 = f1.mean(axis=0), f2.mean(axis=0)
    s1, s2 = np.cov(f1, rowvar=False), np.cov(f2, rowvar=False)
    covmean = linalg.sqrtm(s1 @ s2)
    if not np.isfinite(covmean).all():                       # singular covariances: regularize slightly
        eps_eye = np.eye(s1.shape[0]) * 1e-6
        covmean = linalg.sqrtm((s1 + eps_eye) @ (s2 + eps_eye))
    covmean = covmean.real
    return float(np.sum((mu1 - mu2) ** 2) + np.trace(s1 + s2 - 2.0 * covmean))

@torch.no_grad()
def vae_reconstruct(model, X):
    """Decode the posterior means (no sampling)."""
    model.eval()
    mu, _ = encode_vae(model, X)
    return torch.sigmoid(model.decode(mu))

@torch.no_grad()
def ae_gaussian_samples(model, n, seed=SEED):
    """Decode codes drawn from a Gaussian fitted to the AE codes of the test set (Section 3)."""
    model.eval()
    torch.manual_seed(seed)
    return torch.sigmoid(model.decode(mu32 + sd32 * torch.randn(n, model.latent_dim, device=device)))

N_EVAL = 2_000 if FAST_RUN else 5_000
sample_sets = {
    "real test images": X_test[:N_EVAL],
    "real train images": X_train[:N_EVAL],
    "VAE d=16 reconstructions": vae_reconstruct(vae16, X_test[:N_EVAL]),
    "VAE d=16 prior samples": sample_vae(vae16, N_EVAL),
    "VAE d=2 prior samples": sample_vae(vae2, N_EVAL),
    "beta=4 VAE prior samples": sample_vae(vaes_beta[4.0], N_EVAL),
    "beta=0.25 VAE prior samples": sample_vae(vaes_beta[0.25], N_EVAL),
    "CVAE d=16 samples": sample_cvae(cvae, range(10), n_per_digit=N_EVAL // 10),
    "AE d=32 Gaussian-code samples": ae_gaussian_samples(aes[32], N_EVAL),
}
feats_ref, _, _ = classifier_stats(X_test[5000:5000 + N_EVAL])     # held-out real reference set

print(f"{'image set':32s} {'feature dist.':>13s} {'confidence':>11s} {'class entropy':>14s}")
eval_results = {}
for name, imgs in sample_sets.items():
    feats, conf, ent = classifier_stats(imgs)
    fd = frechet_distance(feats, feats_ref)
    eval_results[name] = (fd, conf, ent)
    print(f"{name:32s} {fd:13.2f} {conf:11.3f} {ent:14.3f}")

The two real sets give the floor of the metric — the distance between two independent samples of real digits is not zero, only small, and any generator scoring close to it is doing very well. Reconstructions come next: they are real digits passed through the bottleneck, so they keep the diversity of the data and lose only sharpness. Among the *generators*, the conditional VAE and the $d = 16$ VAE score best, the $d = 2$ VAE is worse (its samples are generic), and the $\beta = 4$ model is worse still on fidelity while remaining diverse. The autoencoder with Gaussian-fitted codes is by far the worst, with low classifier confidence: its "samples" are the blobs of Section 3. The $\beta = 0.25$ model is instructive — it reconstructs best of all the VAEs, yet its prior samples score poorly, because its aggregate posterior has drifted away from the prior. Reconstruction quality and generation quality are different things, and only a sample-based metric sees the difference.

> **Common pitfall.** Fidelity and diversity trade off, and a single number can hide a collapse. A model that emits one perfect "1" for every prior sample has high confidence, zero class entropy and a large feature distance; always look at samples *and* at more than one metric.

## 8. Applications and modern context

### 8.1 Discrete latents: the VQ-VAE idea

The Gaussian latent space of the VAE is convenient but has two drawbacks: the KL term pushes towards blurry averages, and a continuous code is a poor match for data that is fundamentally discrete or combinatorial (words, or the "which strokes are present" structure of an image). The **vector-quantized VAE** (van den Oord, Vinyals and Kavukcuoglu, 2017) replaces the Gaussian code by a **discrete** one. The encoder outputs a grid of continuous vectors $z_e(x) \in \mathbb{R}^{D}$ (one per spatial position, e.g. $7 \times 7$ of them for MNIST), and each is snapped to its nearest neighbour in a learned **codebook** $\{e_1, \dots, e_K\}$:

$$
z_q(x) = e_k, \qquad k = \arg\min_j \|z_e(x) - e_j\|_2 .
$$

The decoder reconstructs from the quantized grid. The argmin has zero gradient almost everywhere, so the encoder is trained with the **straight-through estimator**: in the backward pass the gradient at $z_q$ is copied unchanged to $z_e$. Two extra terms train the codebook and keep the encoder committed to it; the loss that training minimizes is

$$
\mathcal{L}_{\text{VQ}} = \underbrace{-\log p(x \mid z_q(x))}_{\text{reconstruction NLL}}
\;+\; \underbrace{\|\mathrm{sg}[z_e(x)] - e_k\|^2}_{\text{codebook loss}}
\;+\; \beta\, \underbrace{\|z_e(x) - \mathrm{sg}[e_k]\|^2}_{\text{commitment loss}},
$$

where $\mathrm{sg}[\cdot]$ is the stop-gradient operator (`.detach()` in PyTorch) and $\beta \approx 0.25$. There is no KL term: the prior over code indices is uniform during training and is *learned afterwards* by fitting an autoregressive model (originally a PixelCNN, today a Transformer) to the sequences of code indices of the training images. Generation then means sampling a sequence of indices from the learned prior and decoding it. This two-stage recipe — learn a compressed discrete representation, then model it with a sequence model — is the basis of DALL·E 1, of VQGAN and of most modern audio codecs. Exercise 5 implements the VQ-VAE codebook loss on MNIST.

### 8.2 Latent spaces as the substrate of latent diffusion

The most consequential use of autoencoders today is arguably as a *preprocessing step* for other generative models. Diffusion models (Notebook 12 · GANs and Diffusion Models) generate by iteratively denoising, which is expensive in pixel space at high resolution. **Latent diffusion** (Rombach et al., 2022) first trains an autoencoder — a VAE with a very small KL weight, or a VQ-regularized autoencoder — that compresses a $512 \times 512 \times 3$ image into a $64 \times 64 \times 4$ latent, and then runs the diffusion model in that latent space, 48 times smaller. The autoencoder's job is purely perceptual compression: remove the imperceptible high-frequency detail, keep the semantics. Stable Diffusion and its successors are latent diffusion models, and the weak-KL VAE at their core is a direct descendant of the model we trained in Section 5, trained with additional perceptual and adversarial losses so that its reconstructions are sharp.

### 8.3 Representations for downstream tasks: a linear probe

Finally, are the learned latents *useful* as features? The standard test is a **linear probe**: freeze the encoder, train a logistic regression on its outputs, and measure the accuracy. Linear separability is a strong requirement — it asks that the representation has already done the non-linear work — which is why it is the accepted yard-stick in representation learning. We compare raw pixels, PCA, the autoencoder codes, the posterior means of two VAEs, and, as a supervised reference that has seen the labels, the classifier's own features.

In [ ]:
N_PROBE = 3_000 if FAST_RUN else 10_000
Xp_train = X_train[:N_PROBE]
yp_train = y_train[:N_PROBE].cpu().numpy()

with torch.no_grad():
    Vp = evecs[:, :16]
    probe_feats = {
        "raw pixels (784)": (Xp_train.flatten(1), X_test.flatten(1)),
        "PCA (16)": ((Xp_train.flatten(1) - pix_mean) @ Vp, (X_test.flatten(1) - pix_mean) @ Vp),
        "conv AE codes (32)": (encode_all(aes[32], Xp_train), encode_all(aes[32], X_test)),
        "VAE posterior means (16)": (encode_vae(vae16, Xp_train)[0], encode_vae(vae16, X_test)[0]),
        "beta=4 VAE posterior means (16)": (encode_vae(vaes_beta[4.0], Xp_train)[0],
                                            encode_vae(vaes_beta[4.0], X_test)[0]),
        "supervised CNN features (64)": (torch.as_tensor(classifier_stats(Xp_train)[0]),
                                         torch.as_tensor(classifier_stats(X_test)[0])),
    }

def standardize(train, test):
    """Standardize columns with training statistics; constant columns are left unscaled."""
    m, s = train.mean(axis=0, keepdims=True), train.std(axis=0, keepdims=True)
    s = np.where(s < 1e-3, 1.0, s)
    return (train - m) / s, (test - m) / s

print(f"{'features':34s} {'probe accuracy':>15s}")
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    for name, (ftr, fte) in probe_feats.items():
        ftr, fte = standardize(ftr.cpu().numpy().astype(np.float64), fte.cpu().numpy().astype(np.float64))
        t0 = time.time()
        probe = LogisticRegression(max_iter=300)
        probe.fit(ftr, yp_train)
        acc = probe.score(fte, y_test_np)
        print(f"{name:34s} {acc:15.4f}   ({time.time() - t0:.1f}s)")

A linear classifier on 16 VAE coordinates is far more accurate than one on 16 PCA coordinates and approaches the accuracy on all 784 raw pixels, with a fiftyfold smaller input: the encoder has straightened out much of the class structure without ever seeing a label. The 32-dimensional autoencoder codes do about as well or slightly better — reconstruction is a strong pretext task on MNIST — while the $\beta = 4$ latents, which carry fewer nats, lose a little accuracy. None of them reaches the supervised features, which were shaped by the labels directly. Compare with the contrastive and masked-prediction objectives of Notebook 07 · Transfer Learning and Modern Vision, which close most of that gap on natural images at scale.

## Exercises

**Exercise 1 (warm-up): latent dimension sweep.** Train VAEs with $d \in \{4, 64\}$ (reuse the $d = 2$ and $d = 16$ models), and plot the test reconstruction term, KL term, negative ELBO and the number of active units against $d$. At what point does adding dimensions stop helping, and why?

**Solution to Exercise 1**

In [ ]:
vae_sweep, hist_sweep = {2: vae2, 16: vae16}, {2: hist2, 16: hist16}
for d in [4, 64]:
    torch.manual_seed(SEED)
    print(f"--- VAE with latent dimension {d} ---")
    vae_sweep[d] = ConvVAE(d).to(device)
    t0 = time.time()
    hist_sweep[d] = train_vae(vae_sweep[d], X_train, EPOCHS_VAE, verbose=False)
    print(f"    final test -ELBO {hist_sweep[d]['test_nelbo'][-1]:7.2f} nats | {time.time() - t0:.1f}s")

dims_sorted = sorted(vae_sweep)
sweep_rows = []
print(f"\n{'d':>4s} {'rec NLL':>9s} {'KL':>7s} {'-ELBO':>8s} {'active units':>13s}")
for d in dims_sorted:
    rec, kl, nelbo = evaluate_vae(vae_sweep[d], X_test)
    _, active = latent_usage(vae_sweep[d], X_test)
    sweep_rows.append((rec, kl, nelbo, active))
    print(f"{d:4d} {rec:9.2f} {kl:7.2f} {nelbo:8.2f} {active:13d}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(dims_sorted, [r[0] for r in sweep_rows], "o-", label="reconstruction NLL")
axes[0].plot(dims_sorted, [r[1] for r in sweep_rows], "s-", label="KL")
axes[0].plot(dims_sorted, [r[2] for r in sweep_rows], "^-", label="-ELBO")
axes[0].set_xscale("log", base=2); axes[0].set_xlabel("latent dimension d"); axes[0].set_ylabel("nats per image")
axes[0].set_title("Test objective terms versus d"); axes[0].legend()
axes[1].plot(dims_sorted, [r[3] for r in sweep_rows], "o-", color="C3")
axes[1].plot(dims_sorted, dims_sorted, "--", color="gray", label="all units active")
axes[1].set_xscale("log", base=2); axes[1].set_yscale("log", base=2)
axes[1].set_xlabel("latent dimension d"); axes[1].set_ylabel("active units"); axes[1].legend()
axes[1].set_title("Active units versus d")
plt.tight_layout(); plt.show()

The negative ELBO improves quickly from $d = 2$ to $d = 16$ and then flattens: the model with $d = 64$ is barely better than the one with $d = 16$, and it does not use most of its dimensions — the number of active units saturates well below 64. The reason is the KL penalty: every extra dimension that carries information costs nats, and once the decoder can reconstruct a digit adequately the marginal reconstruction gain of another dimension is smaller than its KL price, so the optimizer switches the dimension off. The VAE thus performs a kind of automatic model selection for its effective latent dimensionality.

**Exercise 2: likelihood-based anomaly scores.** Section 2.7 used the autoencoder's squared error as the anomaly score. Use the $d = 16$ VAE to compute three alternative scores for each test image — the reconstruction NLL at the posterior mean, the KL term, and their sum (a deterministic negative ELBO) — and compare their ROC-AUC on the MNIST-versus-FashionMNIST task. Which term is most discriminative, and why might the KL term alone carry signal?

**Solution to Exercise 2**

In [ ]:
@torch.no_grad()
def vae_anomaly_scores(model, X, batch_size=1000):
    """Per-image reconstruction NLL at z = mu, KL term, and their sum."""
    model.eval()
    rec, kl = [], []
    for (xb,) in iterate_batches([X], batch_size, shuffle=False):
        mu, logvar = model.encode(xb)
        rec.append(recon_bce(model.decode(mu), xb)); kl.append(kl_standard_normal(mu, logvar))
    rec, kl = torch.cat(rec).cpu().numpy(), torch.cat(kl).cpu().numpy()
    return {"VAE reconstruction NLL": rec, "VAE KL term": kl, "VAE -ELBO (rec + KL)": rec + kl}

scores_mnist = vae_anomaly_scores(vae16, X_test)
scores_fashion = vae_anomaly_scores(vae16, X_fashion)
print(f"{'score':26s} {'ROC-AUC':>8s}")
print(f"{'AE squared error (Sec. 2.7)':26s} {auc_ae:8.4f}")
for name in scores_mnist:
    s = np.concatenate([scores_mnist[name], scores_fashion[name]])
    print(f"{name:26s} {roc_auc_score(is_anomaly, s):8.4f}")

The reconstruction NLL of the VAE behaves like the autoencoder's squared error and gives a similar AUC. The KL term alone is also informative: FashionMNIST images are mapped by the encoder to posterior means far from the origin and to unusual variances, because the encoder has learned to place *digits* near the prior and has no idea where to put a sneaker — the KL term measures how "out of distribution" the code is, independently of the pixels. Summing the two terms gives the negative ELBO, a bona fide (bound on the) log-likelihood, and usually the best or near-best score. It is worth remembering, though, that likelihood-based anomaly detection has known failure modes: deep generative models can assign *higher* likelihood to simpler out-of-distribution data than to their training data (Nalisnick et al., 2019), which is the same "low-energy inputs are easy" effect we saw in Section 2.7.

**Exercise 3: a Gaussian decoder with a learned variance.** Replace the Bernoulli decoder by a Gaussian $p_\theta(x \mid z) = \mathcal{N}(x; \hat{x}(z), \sigma^2 I)$ with a single *learned* scalar $\sigma$ (parameterize $\log\sigma$). Write the per-image negative log-likelihood, train a $d = 16$ model with `train_vae` via the `rec_fn` argument, report the learned $\sigma$, compare it with the root-mean-square pixel error of the reconstructions, and look at the KL term and the samples. Explain why the learned $\sigma$ acts like an automatic $\beta$.

**Solution to Exercise 3**

The negative log-likelihood of one image under a Gaussian decoder with $D = 784$ pixels is

$$
-\log p_\theta(x \mid z) = \frac{1}{2\sigma^2}\|x - \hat{x}(z)\|^2 + D \log\sigma + \frac{D}{2}\log(2\pi).
$$

For a fixed decoder the optimal $\sigma^2$ is the mean squared pixel error, and $1/(2\sigma^2)$ is the weight of the squared error relative to the KL. When the reconstructions improve, $\sigma$ shrinks and the effective weight of the reconstruction term grows — so the model behaves like a $\beta$-VAE whose $\beta = 2\sigma^2$ is chosen by maximum likelihood rather than by hand. Note that a Gaussian likelihood is a *density*, so the NLL can be negative when $\sigma$ is small; that is not a bug.

In [ ]:
class GaussianVAE(ConvVAE):
    """VAE whose decoder outputs the mean of a Gaussian with one learned, global standard deviation."""

    def __init__(self, latent_dim, width=WIDTH):
        super().__init__(latent_dim, width)
        self.log_sigma = nn.Parameter(torch.zeros(()))

def make_gaussian_nll(model):
    """Per-image Gaussian negative log-likelihood that reads sigma from the model."""
    def gaussian_nll(logits, x):
        mean = torch.sigmoid(logits)
        sq = (x - mean).pow(2).flatten(1).sum(dim=1)
        D = x[0].numel()
        return 0.5 * sq * torch.exp(-2.0 * model.log_sigma) + D * model.log_sigma + 0.5 * D * math.log(2 * math.pi)
    return gaussian_nll

torch.manual_seed(SEED)
gvae = GaussianVAE(16).to(device)
print("--- VAE with a Gaussian decoder and learned sigma, latent dimension 16 ---")
hist_g = train_vae(gvae, X_train, EPOCHS_VAE, rec_fn=make_gaussian_nll(gvae))

with torch.no_grad():
    sigma = gvae.log_sigma.exp().item()
    rec_imgs = vae_reconstruct(gvae, X_test)
    rmse = (rec_imgs - X_test).pow(2).mean().sqrt().item()
    _, kl_g, _ = evaluate_vae(gvae, X_test, rec_fn=make_gaussian_nll(gvae))
print(f"learned sigma = {sigma:.4f} | RMSE of the reconstructions per pixel = {rmse:.4f} | "
      f"effective beta = 2 sigma^2 = {2 * sigma ** 2:.4f}")
print(f"test KL: Bernoulli VAE {hist16['test_kl'][-1]:.2f} nats | Gaussian VAE with learned sigma {kl_g:.2f} nats")
show_images(sample_vae(gvae, 20), nrow=10, title="Gaussian-decoder VAE with learned sigma: prior samples",
            figsize=(7, 1.9))

The learned $\sigma$ lands close to the root-mean-square pixel error of the reconstructions, as the maximum-likelihood argument predicts. Because $\sigma$ is small, the effective $\beta$ is far below 1: the reconstruction term dominates, the KL term is much larger than in the Bernoulli model (the encoder is allowed to transmit many more nats), and the reconstructions are sharper. The prior samples, on the other hand, tend to be *worse*, for the reason seen in the $\beta$ sweep — a weakly regularized latent space drifts from the prior. Learning $\sigma$ is principled, but for generation one often prefers to cap the effective weight, which is one motivation for the fixed, small KL weight used by the autoencoders of latent diffusion models.

**Exercise 4: the importance-weighted bound.** Burda, Grosse and Salakhutdinov (2016) showed that averaging $K$ importance weights inside the logarithm gives a tighter bound than the ELBO:

$$
\mathcal{L}_K(x) = \mathbb{E}_{z_1, \dots, z_K \sim q_\phi(z \mid x)}\left[\log \frac{1}{K}\sum_{k=1}^{K} \frac{p_\theta(x \mid z_k)\, p(z_k)}{q_\phi(z_k \mid x)}\right], \qquad \mathcal{L}_1 \le \mathcal{L}_K \le \mathcal{L}_{K+1} \le \log p_\theta(x).
$$

Implement `iwae_bound(model, x, K)` for the trained $d = 16$ VAE (compute everything in log space with `torch.logsumexp`) and evaluate it on 1,000 test images (500 under `FAST_RUN`) for $K \in \{1, 5, 20, 100\}$. Check that $K = 1$ agrees with the ELBO from `evaluate_vae` and that the bound increases with $K$.

**Solution to Exercise 4**

In [ ]:
@torch.no_grad()
def iwae_bound(model, x, K):
    """Importance-weighted lower bound on log p(x) with K posterior samples, one value per image."""
    model.eval()
    mu, logvar = model.encode(x)
    std = torch.exp(0.5 * logvar)
    B, d = mu.shape
    eps = torch.randn(K, B, d, device=x.device)
    z = mu[None] + std[None] * eps                                                  # (K, B, d)
    logits = model.decode(z.reshape(K * B, d)).reshape(K, B, *x.shape[1:])
    targets = x[None].expand(K, -1, -1, -1, -1)
    log_px_z = -F.binary_cross_entropy_with_logits(logits, targets, reduction="none").flatten(2).sum(dim=2)
    log_pz = -0.5 * z.pow(2).sum(dim=2) - 0.5 * d * math.log(2 * math.pi)
    log_qz = -0.5 * eps.pow(2).sum(dim=2) - 0.5 * logvar.sum(dim=1)[None] - 0.5 * d * math.log(2 * math.pi)
    log_w = log_px_z + log_pz - log_qz                                              # (K, B)
    return torch.logsumexp(log_w, dim=0) - math.log(K)                              # (B,)

N_IWAE = 500 if FAST_RUN else 1000
x_eval = X_test[:N_IWAE]
torch.manual_seed(SEED)
for K in [1, 5, 20, 100]:
    vals = torch.cat([iwae_bound(vae16, xb, K) for (xb,) in iterate_batches([x_eval], 100, shuffle=False)])
    print(f"K = {K:3d}: importance-weighted bound on log p(x) = {vals.mean().item():8.2f} nats")
print(f"ELBO from evaluate_vae (negated -ELBO):          {-evaluate_vae(vae16, x_eval)[2]:8.2f} nats")

The $K = 1$ estimate matches the ELBO up to Monte Carlo noise, and the bound rises monotonically with $K$, typically by several nats between $K = 1$ and $K = 100$. That gain is a direct measurement of the *posterior approximation gap* of Section 4.2: importance sampling with more particles corrects for the mismatch between the diagonal Gaussian $q_\phi$ and the true posterior. The IWAE paper also *trains* with this objective, which yields better likelihoods; interestingly, Rainforth et al. (2018, *Tighter Variational Bounds Are Not Necessarily Better*) later showed that the tighter bound weakens the gradient signal reaching the encoder, so $q_\phi$ itself can get *worse* as $K$ grows.

**Exercise 5 (challenge): a VQ-VAE on MNIST.** Implement the vector-quantized autoencoder of Section 8.1: a convolutional encoder to a $7 \times 7$ grid of 16-dimensional vectors, a codebook of 64 entries, nearest-neighbour quantization, the straight-through estimator, and the codebook and commitment losses with $\beta = 0.25$. Train it with the per-pixel BCE reconstruction loss, show reconstructions, and report how many codebook entries are in use and the perplexity of the code usage on the test set.

**Solution to Exercise 5**

In [ ]:
class VQVAE(nn.Module):
    """Vector-quantized autoencoder: encoder -> nearest codebook vectors -> decoder, with the VQ losses."""

    def __init__(self, n_codes=64, code_dim=16, width=WIDTH, beta=0.25):
        super().__init__()
        self.beta, self.n_codes = beta, n_codes
        self.enc = nn.Sequential(
            nn.Conv2d(1, width, 4, stride=2, padding=1), nn.ReLU(),                # 28 -> 14
            nn.Conv2d(width, 2 * width, 4, stride=2, padding=1), nn.ReLU(),        # 14 -> 7
            nn.Conv2d(2 * width, code_dim, 1),
        )
        self.codebook = nn.Embedding(n_codes, code_dim)
        self.codebook.weight.data.uniform_(-1.0 / n_codes, 1.0 / n_codes)
        self.dec = nn.Sequential(
            nn.Conv2d(code_dim, 2 * width, 1), nn.ReLU(),
            nn.ConvTranspose2d(2 * width, width, 4, stride=2, padding=1), nn.ReLU(),   # 7 -> 14
            nn.ConvTranspose2d(width, 1, 4, stride=2, padding=1),                      # 14 -> 28
        )

    def quantize(self, z_e):
        """Snap every position of the (B, D, H, W) encoder output to its nearest codebook vector."""
        B, D, H, W = z_e.shape
        flat = z_e.permute(0, 2, 3, 1).reshape(-1, D)                                  # (B*H*W, D)
        idx = torch.cdist(flat, self.codebook.weight).argmin(dim=1)                    # (B*H*W,)
        z_q = self.codebook(idx).reshape(B, H, W, D).permute(0, 3, 1, 2)               # (B, D, H, W)
        return z_q, idx.reshape(B, H, W)

    def forward(self, x):
        z_e = self.enc(x)
        z_q, idx = self.quantize(z_e)
        codebook_loss = F.mse_loss(z_q, z_e.detach())         # moves the codebook towards the encoder outputs
        commitment_loss = F.mse_loss(z_e, z_q.detach())       # keeps the encoder close to its chosen codes
        z_st = z_e + (z_q - z_e).detach()                     # straight-through: forward z_q, backward to z_e
        return self.dec(z_st), codebook_loss + self.beta * commitment_loss, idx

torch.manual_seed(SEED)
vqvae = VQVAE().to(device)
opt = torch.optim.Adam(vqvae.parameters(), lr=2e-3)
EPOCHS_VQ = 1 if FAST_RUN else 8
for epoch in range(1, EPOCHS_VQ + 1):
    vqvae.train(); t0 = time.time(); run_rec = run_vq = 0.0
    for (xb,) in iterate_batches([X_train], BATCH_SIZE):
        logits, vq_loss, _ = vqvae(xb)
        rec = F.binary_cross_entropy_with_logits(logits, xb)   # per-pixel mean, on the scale of the VQ terms
        loss = rec + vq_loss
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        run_rec += rec.item() * xb.shape[0]; run_vq += vq_loss.item() * xb.shape[0]
    print(f"epoch {epoch:2d} | per-pixel BCE {run_rec / X_train.shape[0]:.4f} | VQ loss {run_vq / X_train.shape[0]:.4f} "
          f"| {time.time() - t0:.1f}s")

with torch.no_grad():
    vqvae.eval()
    logits_list, idx_list = [], []
    for (xb,) in iterate_batches([X_test], 1000, shuffle=False):
        lg, _, ix = vqvae(xb)
        logits_list.append(lg); idx_list.append(ix)
    logits, idx = torch.cat(logits_list), torch.cat(idx_list)
    test_bce = recon_bce(logits, X_test).mean().item()
    counts = torch.bincount(idx.flatten(), minlength=vqvae.n_codes).double()
    p_codes = counts / counts.sum()
    perplexity = torch.exp(-(p_codes * torch.log(p_codes + 1e-12)).sum()).item()
print(f"test reconstruction BCE {test_bce:.2f} nats per image (conv AE d=32: {ae_hist[32]['test_bce'][-1]:.2f}) | "
      f"codes in use {(counts > 0).sum().item()} / {vqvae.n_codes} | usage perplexity {perplexity:.1f}")
show_rows([X_test[:10], torch.sigmoid(logits[:10])], ["original", "VQ-VAE"], figsize=(7, 2.3))

The discrete bottleneck reconstructs digits well — each image is now described by 49 integers from a 64-symbol alphabet, i.e. by $49 \log_2 64 = 294$ bits. Two practical observations follow from the code-usage statistics. First, not all codebook entries are used, and the usage perplexity is well below 64: some entries never become the nearest neighbour of any encoder output and receive no gradient, a phenomenon called **codebook collapse**. The usual fixes are exponential-moving-average codebook updates, re-initializing dead codes from recent encoder outputs, or lowering the code dimension. Second, we have a compressor but not yet a generator: sampling requires a prior over the $7 \times 7$ index grids, which is where an autoregressive model or a Transformer enters, as sketched in Section 8.1.

## Summary

- The manifold hypothesis says that natural data occupy a thin, curved, low-dimensional region of input space; on MNIST a few dozen linear components capture most of the variance, and non-linear codes do far better still.
- An undercomplete autoencoder learns a code by reconstruction; with linear maps and squared error it finds exactly the PCA subspace (verified numerically through principal angles and projection matrices), and convolutional autoencoders beat PCA at every bottleneck size by following the manifold's curvature.
- Denoising turns reconstruction into a manifold-learning task and is the conceptual ancestor of score-based diffusion; reconstruction error is a strong out-of-the-box anomaly score, with the caveat that simple inputs are easy to reconstruct.
- Plain autoencoders are poor generators because their latent space has no prior and no smoothness constraint: random codes fall into holes and decode to garbage.
- The VAE is a latent-variable model trained by maximizing the ELBO, $\log p(x) \ge \mathbb{E}_q[\log p(x \mid z)] - \mathrm{KL}(q(z \mid x)\,\|\,p(z))$, whose gap to the log-likelihood is the posterior approximation error; the reparameterization trick makes the encoder trainable with low-variance gradients, and the Gaussian KL has a closed form.
- Monitoring the reconstruction and KL terms separately reveals posterior collapse and inactive units; KL annealing changes the training path, and the $\beta$ weight walks the rate–distortion curve of the model.
- Conditioning both networks on a label yields a conditional VAE that separates identity (given) from style (latent).
- Generative models are evaluated by likelihood bounds, reconstruction metrics and sample-based feature distances such as FID; reconstruction quality and generation quality are different things.
- VAE latents are useful representations (a linear probe on 16 latent coordinates approaches raw-pixel accuracy), discrete codes lead to VQ-VAE and two-stage generators, and low-KL autoencoders are the substrate on which latent diffusion models operate.

## Further reading

- Kingma, D. P. and Welling, M., 2014, *Auto-Encoding Variational Bayes*, ICLR — the original VAE paper.
- Rezende, D. J., Mohamed, S. and Wierstra, D., 2014, *Stochastic Backpropagation and Approximate Inference in Deep Generative Models*, ICML — the reparameterization trick, discovered independently.
- Kingma, D. P. and Welling, M., 2019, *An Introduction to Variational Autoencoders*, Foundations and Trends in Machine Learning — a thorough, readable monograph.
- Burda, Y., Grosse, R. and Salakhutdinov, R., 2016, *Importance Weighted Autoencoders*, ICLR — the IWAE bound and the active-units diagnostic.
- Higgins, I. et al., 2017, *beta-VAE: Learning Basic Visual Concepts with a Constrained Variational Framework*, ICLR; and Alemi, A. et al., 2018, *Fixing a Broken ELBO*, ICML — the rate–distortion view.
- Bowman, S. R. et al., 2016, *Generating Sentences from a Continuous Space*, CoNLL — posterior collapse and KL annealing.
- van den Oord, A., Vinyals, O. and Kavukcuoglu, K., 2017, *Neural Discrete Representation Learning*, NeurIPS — the VQ-VAE.
- Vincent, P. et al., 2008, *Extracting and Composing Robust Features with Denoising Autoencoders*, ICML; and Baldi, P. and Hornik, K., 1989, *Neural Networks and Principal Component Analysis*, Neural Networks — denoising and the linear-AE/PCA result.
- Rombach, R. et al., 2022, *High-Resolution Image Synthesis with Latent Diffusion Models*, CVPR — the VAE inside latent diffusion.
- Goodfellow, I., Bengio, Y. and Courville, A., 2016, *Deep Learning*, MIT Press, chapters 14 (Autoencoders) and 20 (Deep Generative Models); Murphy, K. P., 2023, *Probabilistic Machine Learning: Advanced Topics*, MIT Press, chapter on VAEs; and the PyTorch documentation, https://docs.pytorch.org/docs/stable/ (`torch.distributions`, `nn.ConvTranspose2d`, `F.binary_cross_entropy_with_logits`).